In [ ]:
#@title 1️⃣ Install Dependencies
!apt-get install -y libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1
!pip install -q fastapi "uvicorn[standard]" python-multipart websockets nest_asyncio
!pip install -q "opencv-python-headless==4.10.0.84" Pillow numpy
!pip install -q "transformers>=4.41.0" torch torchvision accelerate
!pip install -q sentencepiece protobuf qwen-vl-utils
print('✅ All packages installed')


In [ ]:
#@title 2️⃣ Write OCR Engine (v12 — Precise Current Frame + Server-Side Accumulated Scan)
import pathlib

ocr_engine_code = 'import cv2, numpy as np, torch, re, time, warnings\nfrom PIL import Image\nwarnings.filterwarnings("ignore")\n\nMODEL = None\nPROCESSOR = None\nDEVICE_INFO = "unknown"\n\n\ndef load_model():\n    global MODEL, PROCESSOR, DEVICE_INFO\n    if MODEL is not None:\n        return MODEL, PROCESSOR\n    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor\n    if not torch.cuda.is_available():\n        print("WARNING: No GPU found! CPU will be VERY slow (60-120s/image).")\n        print("   Runtime -> Change runtime type -> T4 GPU, then restart & rerun.")\n        DTYPE = torch.float32\n    else:\n        print(f"GPU: {torch.cuda.get_device_name(0)}")\n        DTYPE = torch.float16\n    print("Loading Qwen2-VL-2B-Instruct…")\n    MODEL = Qwen2VLForConditionalGeneration.from_pretrained(\n        "Qwen/Qwen2-VL-2B-Instruct", torch_dtype=DTYPE, device_map="auto")\n    PROCESSOR = AutoProcessor.from_pretrained(\n        "Qwen/Qwen2-VL-2B-Instruct",\n        min_pixels=256*28*28, max_pixels=1280*28*28)\n    DEVICE_INFO = str(next(MODEL.parameters()).device)\n    print(f"Qwen2-VL ready on {DEVICE_INFO}!")\n    return MODEL, PROCESSOR\n\n\n# ── Preprocessor ─────────────────────────────────────────────────────────────\n\nclass Preprocessor:\n    def load_bytes(self, image_bytes):\n        arr = np.frombuffer(image_bytes, np.uint8)\n        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)\n        if img is None:\n            raise ValueError("Cannot decode image")\n        return img\n\n    def upscale(self, img, min_dim=1200):\n        h, w = img.shape[:2]\n        if min(h, w) < min_dim:\n            s = min_dim / min(h, w)\n            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_CUBIC)\n        return img\n\n    def remove_shadows(self, img):\n        planes = []\n        for p in cv2.split(img):\n            bg = cv2.medianBlur(cv2.dilate(p, np.ones((7,7), np.uint8)), 21)\n            planes.append(cv2.normalize(255 - cv2.absdiff(p, bg), None, 0, 255, cv2.NORM_MINMAX))\n        return cv2.merge(planes)\n\n    def auto_rotate(self, img):\n        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n        edges = cv2.Canny(cv2.GaussianBlur(gray,(5,5),0), 50, 150, apertureSize=3)\n        lines = cv2.HoughLines(edges, 1, np.pi/180, threshold=80)\n        if lines is None: return img, 0.0\n        angles = [float(np.degrees(l[0][1])-90) for l in lines[:60]\n                  if -45 < float(np.degrees(l[0][1])-90) < 45]\n        if not angles: return img, 0.0\n        angle = float(np.median(angles))\n        if abs(angle) < 0.5: return img, 0.0\n        h, w = img.shape[:2]; cx, cy = w//2, h//2\n        M = cv2.getRotationMatrix2D((cx,cy), angle, 1.0)\n        cos, sin = abs(M[0,0]), abs(M[0,1])\n        nw, nh = int(h*sin+w*cos), int(h*cos+w*sin)\n        M[0,2] += nw/2-cx; M[1,2] += nh/2-cy\n        return cv2.warpAffine(img, M, (nw,nh), flags=cv2.INTER_CUBIC,\n                              borderMode=cv2.BORDER_REPLICATE), angle\n\n    def enhance(self, img):\n        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)\n        l, a, b = cv2.split(lab)\n        cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)\n        img = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)\n        return cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))\n\n    def denoise(self, img):\n        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)\n\n    def run_full(self, img):\n        img = self.upscale(img)\n        img = self.remove_shadows(img)\n        img, angle = self.auto_rotate(img)\n        img = self.denoise(img)\n        img = self.enhance(img)\n        return img, angle\n\n    # v12 IMPROVED: higher cap (1280), upscale small frames, adaptive sharpening,\n    # bilateral filter to reduce sensor noise while keeping edges crisp\n    def run_fast(self, img, max_dim=1280):\n        h, w = img.shape[:2]\n        # Upscale tiny frames so model sees enough detail\n        if max(h, w) < 640:\n            s = 640 / max(h, w)\n            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_CUBIC)\n            h, w = img.shape[:2]\n        if max(h, w) > max_dim:\n            s = max_dim / max(h, w)\n            img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)\n        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()\n        # Adaptive sharpen: stronger kernel for blurrier frames\n        if lap_var < 80:\n            img = cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,6,-1],[0,-1,0]]))\n        elif lap_var < 200:\n            img = cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))\n        # CLAHE contrast enhancement\n        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)\n        l, a, b = cv2.split(lab)\n        cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)\n        img = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)\n        # Bilateral: removes noise, preserves text edges\n        img = cv2.bilateralFilter(img, 5, 40, 40)\n        return img, 0.0\n\n\n# ── Document Detector (v12: both methods fully implemented) ───────────────────\n\nclass DocumentDetector:\n    def _color_crop(self, img):\n        hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)\n        mask = cv2.inRange(hsv, np.array([0,0,150]), np.array([180,80,255]))\n        k    = np.ones((15,15), np.uint8)\n        mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)\n        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n        if not cnts: return None\n        largest = max(cnts, key=cv2.contourArea)\n        if cv2.contourArea(largest) < img.shape[0]*img.shape[1]*0.10: return None\n        x, y, w, h = cv2.boundingRect(largest); pad = 15\n        x = max(0, x-pad); y = max(0, y-pad)\n        w = min(img.shape[1]-x, w+2*pad); h = min(img.shape[0]-y, h+2*pad)\n        return img[y:y+h, x:x+w]\n\n    def _edge_crop(self, img):\n        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n        edges = cv2.dilate(cv2.Canny(cv2.GaussianBlur(gray,(5,5),0), 30, 100),\n                           np.ones((3,3), np.uint8), iterations=2)\n        cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n        area = img.shape[0] * img.shape[1]\n        for cnt in sorted(cnts, key=cv2.contourArea, reverse=True)[:8]:\n            peri   = cv2.arcLength(cnt, True)\n            approx = cv2.approxPolyDP(cnt, 0.02*peri, True)\n            if len(approx) == 4 and cv2.contourArea(cnt) > area * 0.15:\n                x, y, w, h = cv2.boundingRect(cnt); pad = 15\n                x = max(0, x-pad); y = max(0, y-pad)\n                w = min(img.shape[1]-x, w+2*pad); h = min(img.shape[0]-y, h+2*pad)\n                return img[y:y+h, x:x+w]\n        return None\n\n    def extract(self, img):\n        crop = self._color_crop(img)\n        if crop is not None: return crop\n        crop = self._edge_crop(img)\n        if crop is not None: return crop\n        return img\n\n\n# ── Prompts ───────────────────────────────────────────────────────────────────\n\n# Two complementary prompts for uploaded images — best-of-2 by score\nIMAGE_PROMPTS = [\n    (\n        "Perform high-precision OCR on this image. Transcribe ALL text exactly as it appears.\\n"\n        "Rules:\\n"\n        "- Copy every character, digit and symbol exactly — NO spelling corrections.\\n"\n        "- Preserve line breaks exactly as in the source.\\n"\n        "- For forms / receipts: format as \'Label: Value\' one per line.\\n"\n        "- For tables: separate columns with \' | \'.\\n"\n        "- Numbers, dates, currency must be character-for-character accurate.\\n"\n        "- Do NOT interpret, summarise, or add any text not physically present.\\n"\n        "Output ONLY the transcribed text."\n    ),\n    (\n        "Read every piece of text visible in this image.\\n"\n        "If it is a form or receipt: output \'Field: Value\' pairs, one per line.\\n"\n        "If it is a paragraph or printed document: preserve exact line structure.\\n"\n        "Copy all characters precisely — do not fix grammar or spelling.\\n"\n        "If a character is unclear, write your best guess.\\n"\n        "Output only the extracted text, nothing else."\n    ),\n]\n\n# v12 IMPROVED camera prompt — stricter, more explicit line-break rules\nCAMERA_PROMPT = (\n    "You are a high-precision OCR scanner. "\n    "Transcribe ONLY the text physically visible in this image.\\n"\n    "STRICT RULES — follow every one:\\n"\n    "1. Copy every word and number EXACTLY as written — zero corrections.\\n"\n    "2. One source line = one output line. Preserve ALL line breaks.\\n"\n    "3. Numbered or bulleted items each get their OWN separate line.\\n"\n    "4. Key-value pairs (e.g. \'Name: John\') stay on a single line.\\n"\n    "5. Unclear characters → write best guess; mark truly unreadable with [?].\\n"\n    "6. If blurry, too dark, or NO readable text → output exactly: [No text found]\\n"\n    "7. NEVER add explanations, commentary, or invented words.\\n"\n    "Output ONLY the transcribed text."\n)\n\n\n# ── VLM Runner ────────────────────────────────────────────────────────────────\n\ndef run_vlm(img_bgr, prompt, max_new_tokens=512):\n    model, processor = load_model()\n    pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))\n    messages = [{"role":"user","content":[\n        {"type":"image","image":pil_img},\n        {"type":"text","text":prompt}\n    ]}]\n    text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs  = processor(text=[text_in], images=[pil_img], padding=True,\n                        return_tensors="pt").to(model.device)\n    with torch.no_grad():\n        ids = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            temperature=None,\n            top_p=None,\n            repetition_penalty=1.05,  # reduces repeated-line hallucinations\n        )\n    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, ids)]\n    return processor.batch_decode(trimmed, skip_special_tokens=True,\n                                  clean_up_tokenization_spaces=False)[0].strip()\n\n\n# ── Scoring ───────────────────────────────────────────────────────────────────\n\ndef _score(text):\n    if not text or text.strip() in ("[No text found]", "[No text detected]"):\n        return 0\n    lines         = [l for l in text.splitlines() if l.strip()]\n    real_words    = [w for w in text.split() if len(w)>1 and sum(c.isalpha() for c in w)/len(w)>0.5]\n    number_hits   = len(re.findall(r"\\b\\d[\\d.,]+\\b", text))\n    currency_hits = len(re.findall(r"\\d+\\s*[€£$₹]", text))\n    garbage       = sum(1 for c in text if not c.isalnum()\n                        and c not in " \\n\\t.,;:!?€$£₹%-_()\'\\"- /|[]")\n    garbage_ratio = garbage / max(len(text), 1)\n    if garbage_ratio > 0.25:\n        return -999\n    unique_ln = len(set(lines))\n    repeats   = (len(lines) - unique_ln) * 20\n    return (len(real_words)*4 + number_hits*3 + currency_hits*10\n            + unique_ln*3 - int(garbage_ratio*150) - repeats)\n\n\n# ── String-similarity helpers ─────────────────────────────────────────────────\n\ndef _levenshtein_ratio(a, b):\n    if not a and not b: return 1.0\n    if not a or not b:  return 0.0\n    la, lb = len(a), len(b)\n    if abs(la - lb) / max(la, lb) > 0.65: return 0.0\n    prev = list(range(lb + 1))\n    for i, ca in enumerate(a):\n        curr = [i + 1] + [0] * lb\n        for j, cb in enumerate(b):\n            curr[j+1] = min(prev[j] + (0 if ca == cb else 1),\n                            curr[j] + 1, prev[j+1] + 1)\n        prev = curr\n    return 1.0 - prev[lb] / max(la, lb)\n\n\ndef _ngram_sim(a, b, n=3):\n    if len(a) < n and len(b) < n: return 1.0 if a == b else 0.0\n    na = set(a[i:i+n] for i in range(len(a)-n+1))\n    nb = set(b[i:i+n] for i in range(len(b)-n+1))\n    if not na or not nb: return 0.0\n    inter = len(na & nb)\n    return inter / (len(na) + len(nb) - inter)\n\n\ndef _norm_str(text):\n    text = text.lower()\n    text = re.sub(r"^\\s*[\\d()\\[\\]]+[.)\\]]?\\s*", "", text)\n    text = re.sub(r"(\\d)[,\\.](\\d)", r"\\1\\2", text)\n    text = re.sub(r"[^a-z0-9 ]", "", text)\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef _line_sim(a, b):\n    na, nb = _norm_str(a), _norm_str(b)\n    if not na and not nb: return 1.0\n    if na == nb: return 1.0\n    lev = _levenshtein_ratio(na, nb)\n    tri = _ngram_sim(na, nb, 3)\n    bi  = _ngram_sim(na, nb, 2) if min(len(na), len(nb)) < 12 else 0.0\n    return max(lev, tri, bi)\n\n\ndef _is_hallucination(line):\n    if len(line) > 100: return True\n    bad = [\n        "connection call", "will make", "can be used", "is used",\n        "refers to", "stands for", "the following", "as follows",\n        "in this image", "the text shows", "the image shows",\n        "it appears", "this is a", "note that", "the image contains",\n    ]\n    ll = line.lower()\n    return any(p in ll for p in bad)\n\n\n# ── Within-frame deduplication ────────────────────────────────────────────────\n\ndef deduplicate_lines(text, threshold=0.72):\n    """Remove near-duplicate lines within a single OCR result."""\n    lines = [l.strip() for l in text.splitlines() if l.strip()]\n    seen_norms, result = [], []\n    for line in lines:\n        if _is_hallucination(line): continue\n        n = _norm_str(line)\n        if not n or len(n) < 3: continue\n        if n in seen_norms: continue\n        if any(_line_sim(n, s) >= threshold for s in seen_norms): continue\n        seen_norms.append(n)\n        result.append(line)\n    return "\\n".join(result)\n\n\n# ── Accumulated Scan Merger (SERVER-SIDE) ─────────────────────────────────────\n\ndef merge_into_accumulated(accumulated: str, new_frame: str,\n                            threshold: float = 0.75) -> str:\n    """\n    Merge new_frame text into accumulated, appending only lines that are\n    genuinely new (not already captured via fuzzy matching).\n    Preserves order: accumulated lines first, new unique lines appended.\n    """\n    if not new_frame or new_frame.strip() in (\n            "[No text found]", "[No text detected]", ""):\n        return accumulated\n\n    acc_lines = [l.strip() for l in accumulated.splitlines() if l.strip()]\n    new_lines = [l.strip() for l in new_frame.splitlines() if l.strip()]\n    acc_norms = [_norm_str(al) for al in acc_lines]\n\n    for new_line in new_lines:\n        n = _norm_str(new_line)\n        if not n or len(n) < 3: continue\n        if _is_hallucination(new_line): continue\n        already = any(_line_sim(n, an) >= threshold for an in acc_norms if an)\n        if not already:\n            acc_lines.append(new_line)\n            acc_norms.append(n)\n\n    return "\\n".join(acc_lines)\n\n\n# ── Public API ────────────────────────────────────────────────────────────────\n\npreprocessor = Preprocessor()\ndetector     = DocumentDetector()\nMIN_CAMERA_SCORE = 25   # v12: slightly more permissive than v11\'s 28\n\n\ndef ocr_image_bytes(image_bytes, auto_crop=True, lang_hint=None):\n    """Full-quality OCR for uploaded images. Runs both prompts, picks best score."""\n    t0  = time.time()\n    raw = preprocessor.load_bytes(image_bytes)\n    if auto_crop:\n        raw = detector.extract(raw)\n    img, angle = preprocessor.run_full(raw)\n\n    best_text, best_score = "", -999\n    for prompt in IMAGE_PROMPTS:\n        text = run_vlm(img, prompt)\n        s    = _score(text)\n        if s > best_score:\n            best_text, best_score = text, s\n\n    cleaned = deduplicate_lines(best_text) if best_text else "[No text detected]"\n    return {\n        "text":            cleaned,\n        "score":           best_score,\n        "angle_corrected": round(angle, 2),\n        "elapsed":         round(time.time()-t0, 2),\n        "device":          DEVICE_INFO,\n    }\n\n\ndef ocr_camera_frame(image_bytes):\n    """\n    Fast OCR for a single live camera frame.\n    Returns current_text + score. Accumulated merging is done in the WS handler.\n    """\n    t0  = time.time()\n    raw = preprocessor.load_bytes(image_bytes)\n    img, _ = preprocessor.run_fast(raw, max_dim=1280)\n    text   = run_vlm(img, CAMERA_PROMPT, max_new_tokens=400)\n    score  = _score(text)\n    elapsed = round(time.time()-t0, 2)\n\n    if score < MIN_CAMERA_SCORE:\n        return {"current_text": "", "score": score,\n                "elapsed": elapsed, "device": DEVICE_INFO, "low_conf": True}\n\n    cleaned = deduplicate_lines(text)\n    return {"current_text": cleaned or "[No text found]", "score": score,\n            "elapsed": elapsed, "device": DEVICE_INFO, "low_conf": False}\n'

pathlib.Path("ocr_engine.py").write_text(ocr_engine_code)
print("✅ ocr_engine.py written (v12)")


In [ ]:
#@title 3️⃣ Write FastAPI Backend (v13.1 — Accumulated Edit + Toggle + Cloudflare /docs fix)
import pathlib

main_code = "import asyncio, base64, json, os, time\nfrom collections import defaultdict, deque\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\nfrom typing import List, Optional\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport uvicorn\nfrom fastapi import FastAPI, File, Form, Request, UploadFile, WebSocket, WebSocketDisconnect\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import HTMLResponse, JSONResponse\nfrom starlette.middleware.base import BaseHTTPMiddleware\n\nexecutor = ThreadPoolExecutor(max_workers=2)\n\n\nclass CSPMiddleware(BaseHTTPMiddleware):\n    async def dispatch(self, request, call_next):\n        response = await call_next(request)\n        response.headers[\"Content-Security-Policy\"] = (\n            \"default-src 'self'; \"\n            \"script-src 'self' 'unsafe-inline' cdn.jsdelivr.net cdnjs.cloudflare.com; \"\n            \"style-src 'self' 'unsafe-inline' cdn.jsdelivr.net cdnjs.cloudflare.com; \"\n            \"img-src 'self' data: blob:; \"\n            \"connect-src 'self' ws: wss:; \"\n            \"media-src 'self' blob:;\"\n        )\n        return response\n\n\nclass _RateLimiter:\n    def __init__(self):\n        self._windows = defaultdict(deque)\n\n    def allow(self, key, max_requests, window_sec=60.0):\n        now = time.monotonic()\n        dq  = self._windows[key]\n        while dq and dq[0] < now - window_sec:\n            dq.popleft()\n        if len(dq) >= max_requests:\n            return False\n        dq.append(now)\n        return True\n\n    def remaining(self, key, max_requests, window_sec=60.0):\n        now = time.monotonic()\n        dq  = self._windows[key]\n        while dq and dq[0] < now - window_sec:\n            dq.popleft()\n        return max(0, max_requests - len(dq))\n\n\nRL = _RateLimiter()\nLIMIT_IMAGE = 20\nLIMIT_BATCH = 6\n\n\ndef _client_ip(request):\n    xff = request.headers.get(\"x-forwarded-for\")\n    return (xff.split(\",\")[0].strip() if xff else request.client.host) or \"unknown\"\n\n\ndef _rate_check(request, limit):\n    ip  = _client_ip(request)\n    key = f\"{ip}:{request.url.path}\"\n    if not RL.allow(key, limit):\n        rem = 60 - int(time.monotonic() % 60)\n        return JSONResponse(\n            status_code=429,\n            content={\"error\": f\"Rate limit exceeded ({limit} req/min). Retry in ~{rem}s.\",\n                     \"retry_after\": rem},\n            headers={\"Retry-After\": str(rem)},\n        )\n    return None\n\n\n@asynccontextmanager\nasync def lifespan(app):\n    loop = asyncio.get_event_loop()\n    await loop.run_in_executor(executor, _preload_model)\n    yield\n\n\ndef _preload_model():\n    try:\n        from ocr_engine import load_model\n        load_model()\n    except Exception as e:\n        print(f\"Model preload failed: {e}\")\n\n\n# \u2500\u2500 Root path detection for Cloudflare tunnel (/docs fix) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# When running behind Cloudflare, the public URL is HTTPS but uvicorn sees HTTP.\n# Setting root_path allows FastAPI/Swagger to generate correct URLs.\nROOT_PATH = os.environ.get(\"ROOT_PATH\", \"\")\n\napp = FastAPI(\n    title=\"OCR API\",\n    version=\"4.1.0\",\n    lifespan=lifespan,\n    root_path=ROOT_PATH,\n    docs_url=\"/docs\",\n    redoc_url=\"/redoc\",\n    openapi_url=\"/openapi.json\",\n)\napp.add_middleware(CORSMiddleware, allow_origins=[\"*\"],\n                   allow_credentials=True, allow_methods=[\"*\"], allow_headers=[\"*\"])\napp.add_middleware(CSPMiddleware)\n\nSTATIC_DIR = Path(__file__).parent / \"static\"\nSTATIC_DIR.mkdir(exist_ok=True)\n\n\n@app.get(\"/\", response_class=HTMLResponse)\nasync def serve_ui():\n    html_path = STATIC_DIR / \"index.html\"\n    if html_path.exists():\n        return HTMLResponse(content=html_path.read_text(encoding=\"utf-8\"))\n    return HTMLResponse(content=\"<h2>UI not found. Run Cell 4 first.</h2>\")\n\n\n@app.get(\"/health\")\nasync def health(request: Request):\n    import torch\n    from ocr_engine import DEVICE_INFO\n    ip = _client_ip(request)\n    return {\n        \"status\":    \"ok\",\n        \"gpu\":       torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU (slow!)\",\n        \"device\":    DEVICE_INFO,\n        \"timestamp\": time.time(),\n        \"rate_limits\": {\n            \"image_remaining\": RL.remaining(f\"{ip}:/ocr/image\", LIMIT_IMAGE),\n            \"batch_remaining\": RL.remaining(f\"{ip}:/ocr/batch\", LIMIT_BATCH),\n        },\n    }\n\n\n@app.post(\"/ocr/image\")\nasync def ocr_single_image(request: Request,\n                            file: UploadFile = File(...),\n                            auto_crop: bool  = Form(True),\n                            lang_hint: Optional[str] = Form(None)):\n    if (err := _rate_check(request, LIMIT_IMAGE)): return err\n    from ocr_engine import ocr_image_bytes\n    image_bytes = await file.read()\n    loop   = asyncio.get_event_loop()\n    result = await loop.run_in_executor(executor, ocr_image_bytes, image_bytes, auto_crop, lang_hint)\n    result[\"filename\"] = file.filename\n    return JSONResponse(content=result)\n\n\n@app.post(\"/ocr/batch\")\nasync def ocr_batch_images(request: Request,\n                            files: List[UploadFile] = File(...),\n                            auto_crop: bool          = Form(True),\n                            lang_hint: Optional[str] = Form(None)):\n    if (err := _rate_check(request, LIMIT_BATCH)): return err\n    from ocr_engine import ocr_image_bytes\n    loop    = asyncio.get_event_loop()\n    results = []\n    for f in files:\n        try:\n            image_bytes = await f.read()\n            result = await loop.run_in_executor(executor, ocr_image_bytes, image_bytes, auto_crop, lang_hint)\n            result[\"filename\"] = f.filename\n            results.append(result)\n        except Exception as e:\n            results.append({\"filename\": f.filename, \"error\": str(e),\n                            \"text\": \"\", \"score\": 0, \"elapsed\": 0})\n    return JSONResponse(content={\"results\": results, \"count\": len(results)})\n\n\nMIN_FRAME_GAP = 1.0\n\n\n@app.websocket(\"/ocr/camera\")\nasync def ocr_camera_ws(websocket: WebSocket):\n    \"\"\"\n    v13 WebSocket \u2014 per-session server-side accumulation with edit support.\n\n    Accepted messages:\n      {\"frame\": \"<base64 JPEG>\"}              \u2014 process a camera frame\n      {\"frame\": ..., \"accumulate\": false}     \u2014 process but skip accumulation\n      {\"command\": \"clear\"}                    \u2014 reset accumulated_text\n      {\"command\": \"set_accumulated\", \"text\": \"...\"} \u2014 overwrite with edited text\n\n    Sent messages always include BOTH:\n      current_text     \u2014 what was read from this specific frame\n      accumulated_text \u2014 all unique lines seen so far this session\n    \"\"\"\n    from ocr_engine import ocr_camera_frame, merge_into_accumulated\n    await websocket.accept()\n\n    loop             = asyncio.get_event_loop()\n    frame_id         = 0\n    processing       = False\n    last_frame_time  = 0.0\n    accumulated_text = \"\"   # Per-session accumulated scan state\n\n    try:\n        while True:\n            try:\n                raw = await asyncio.wait_for(websocket.receive_text(), timeout=60.0)\n            except asyncio.TimeoutError:\n                await websocket.send_json({\"heartbeat\": True})\n                continue\n\n            data = json.loads(raw)\n\n            # \u2500\u2500 Clear command \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            if data.get(\"command\") == \"clear\":\n                accumulated_text = \"\"\n                await websocket.send_json({\n                    \"command_ack\":     \"cleared\",\n                    \"accumulated_text\": \"\",\n                    \"frame_id\":        frame_id,\n                })\n                continue\n\n            # \u2500\u2500 Set accumulated (edit from frontend) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            if data.get(\"command\") == \"set_accumulated\":\n                accumulated_text = data.get(\"text\", \"\")\n                await websocket.send_json({\n                    \"command_ack\":     \"set_accumulated\",\n                    \"accumulated_text\": accumulated_text,\n                    \"frame_id\":        frame_id,\n                })\n                continue\n\n            # \u2500\u2500 Frame processing \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n            now = time.monotonic()\n\n            if processing:\n                await websocket.send_json({\n                    \"current_text\":    \"Processing\u2026\",\n                    \"accumulated_text\": accumulated_text,\n                    \"elapsed\":         0,\n                    \"frame_id\":        frame_id,\n                    \"busy\":            True,\n                })\n                continue\n\n            gap = now - last_frame_time\n            if gap < MIN_FRAME_GAP:\n                await websocket.send_json({\n                    \"current_text\":    f\"Too fast \u2014 wait {MIN_FRAME_GAP - gap:.1f}s\",\n                    \"accumulated_text\": accumulated_text,\n                    \"elapsed\":         0,\n                    \"frame_id\":        frame_id,\n                    \"busy\":            True,\n                    \"rate_limited\":    True,\n                })\n                continue\n\n            b64 = data.get(\"frame\", \"\")\n            if not b64:\n                await websocket.send_json({\"error\": \"empty frame\"})\n                continue\n\n            # Whether to merge into accumulated (client-controlled toggle)\n            do_accumulate = data.get(\"accumulate\", True)\n\n            if \",\" in b64:\n                b64 = b64.split(\",\", 1)[1]\n            image_bytes     = base64.b64decode(b64)\n            frame_id       += 1\n            processing      = True\n            last_frame_time = now\n\n            try:\n                result = await loop.run_in_executor(executor, ocr_camera_frame, image_bytes)\n\n                current_text = result.get(\"current_text\", \"\")\n                low_conf     = result.get(\"low_conf\", False)\n\n                # Only merge into accumulated on confident reads AND if enabled\n                if current_text and not low_conf and do_accumulate:\n                    accumulated_text = merge_into_accumulated(accumulated_text, current_text)\n\n                display_current = current_text\n                if not display_current:\n                    display_current = (\n                        \"[Low confidence \u2014 move closer or improve lighting]\"\n                        if low_conf else \"[No text found]\"\n                    )\n\n                await websocket.send_json({\n                    \"current_text\":    display_current,\n                    \"accumulated_text\": accumulated_text,\n                    \"score\":           result.get(\"score\", 0),\n                    \"elapsed\":         result.get(\"elapsed\", 0),\n                    \"frame_id\":        frame_id,\n                    \"device\":          result.get(\"device\", \"\"),\n                    \"low_conf\":        low_conf,\n                    \"busy\":            False,\n                })\n            finally:\n                processing = False\n\n    except WebSocketDisconnect:\n        print(f\"Camera WS disconnected after {frame_id} frames\")\n    except Exception as e:\n        print(f\"Camera WS error: {e}\")\n        try:\n            await websocket.send_json({\"error\": str(e)})\n        except Exception:\n            pass\n\n\nif __name__ == \"__main__\":\n    PORT = int(os.environ.get(\"PORT\", 8000))\n    uvicorn.run(\"main:app\", host=\"0.0.0.0\", port=PORT, reload=False, log_level=\"info\")\n"

pathlib.Path("main.py").write_text(main_code)
print("✅ main.py written (v13)")


In [ ]:
#@title 3️⃣b  ➕ Patch main.py — Add HTTP Camera Frame Endpoint
# Adds a stateless REST endpoint as alternative to the WebSocket camera.
# Client sends each frame + current accumulated text; server returns merged OCR text.
import pathlib

src = pathlib.Path("main.py").read_text()

_EP = [
    "\n\n",
    "@app.post(\"/ocr/camera/frame\")\n",
    "async def ocr_camera_frame_http(\n",
    "    request:     Request,\n",
    "    file:        UploadFile = File(...),\n",
    "    accumulated: str        = Form(\"\"),\n",
    "):\n",
    "    \"\"\"\n",
    "    HTTP REST alternative to the WebSocket camera endpoint (stateless).\n",
    "    The client manages accumulated text locally and sends it with each frame.\n",
    "\n",
    "    Form fields\n",
    "    -----------\n",
    "    file        : JPEG / PNG / WEBP image of the camera frame (required)\n",
    "    accumulated : accumulated OCR text from previous frames   (optional)\n",
    "    \"\"\"\n",
    "    if (err := _rate_check(request, LIMIT_IMAGE)):\n",
    "        return err\n",
    "    from ocr_engine import ocr_camera_frame, merge_into_accumulated\n",
    "    image_bytes = await file.read()\n",
    "    loop   = asyncio.get_event_loop()\n",
    "    result = await loop.run_in_executor(executor, ocr_camera_frame, image_bytes)\n",
    "\n",
    "    current  = result.get(\"current_text\", \"\")\n",
    "    low_conf = result.get(\"low_conf\", False)\n",
    "\n",
    "    new_acc = (\n",
    "        merge_into_accumulated(accumulated, current)\n",
    "        if current and not low_conf\n",
    "        else accumulated\n",
    "    )\n",
    "\n",
    "    if not current:\n",
    "        current = (\n",
    "            \"[Low confidence \u2014 move closer or improve lighting]\"\n",
    "            if low_conf else \"[No text found]\"\n",
    "        )\n",
    "\n",
    "    return JSONResponse(content={\n",
    "        \"current_text\":     current,\n",
    "        \"accumulated_text\": new_acc,\n",
    "        \"score\":            result.get(\"score\",   0),\n",
    "        \"elapsed\":          result.get(\"elapsed\", 0),\n",
    "        \"device\":           result.get(\"device\",  \"\"),\n",
    "        \"low_conf\":         low_conf,\n",
    "    })\n",
]

HTTP_ENDPOINT = "".join(_EP)

marker = '\nif __name__ == "__main__":'
idx = src.find(marker)
if idx == -1:
    raise RuntimeError("main.py insertion point not found — did main.py write fail?")

new_src = src[:idx] + HTTP_ENDPOINT + src[idx:]
pathlib.Path("main.py").write_text(new_src)
print("\u2705 main.py patched: POST /ocr/camera/frame endpoint added")
print("   Accepts: file (image) + accumulated (optional text from prev frames)")


In [ ]:
#@title 4️⃣ Write Frontend HTML (v13.1 — Edit fix: sync serverAccumulated from DOM)
import pathlib

html_code = "<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"UTF-8\"/>\n<meta name=\"viewport\" content=\"width=device-width,initial-scale=1.0\"/>\n<title>OCR Studio \u2014 Qwen2-VL v13</title>\n<style>\n*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}\n:root{\n  --bg:#0d1117;--surface:#161b22;--surface2:#21262d;--border:#30363d;\n  --accent:#58a6ff;--accent2:#7ee787;--text:#e6edf3;--muted:#8b949e;\n  --red:#f85149;--warn:#f0883e;--purple:#d2a8ff;--cyan:#79c0ff;\n  --radius:12px;--shadow:0 8px 32px rgba(0,0,0,.6);\n}\nhtml,body{height:100%;background:var(--bg);color:var(--text);font-family:'Segoe UI',system-ui,sans-serif;font-size:15px}\n.app{display:flex;flex-direction:column;min-height:100vh}\nheader{background:var(--surface);border-bottom:1px solid var(--border);padding:14px 28px;display:flex;align-items:center;gap:14px;flex-wrap:wrap}\nheader h1{font-size:1.25rem;font-weight:700;color:var(--accent)}\n.badge{border-radius:20px;padding:3px 12px;font-size:.75rem;border:1px solid var(--border);background:var(--surface2);color:var(--accent2)}\n.badge.warn{border-color:var(--warn);color:var(--warn)}\n.badge.err{border-color:var(--red);color:var(--red)}\nmain{flex:1;max-width:1400px;width:100%;margin:0 auto;padding:20px;display:flex;flex-direction:column;gap:18px}\n.warn-banner{background:rgba(240,136,62,.12);border:1px solid var(--warn);border-radius:10px;padding:14px 18px;color:var(--warn);font-size:.9rem;display:none}\n.warn-banner.show{display:block}\n.tabs{display:flex;gap:8px;flex-wrap:wrap}\n.tab{padding:8px 22px;border-radius:8px;border:1px solid var(--border);background:var(--surface);cursor:pointer;font-size:.9rem;color:var(--muted);transition:all .2s;user-select:none}\n.tab:hover{border-color:var(--accent);color:var(--text)}\n.tab.active{background:var(--accent);border-color:var(--accent);color:#000;font-weight:600}\n.card{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:22px;box-shadow:var(--shadow)}\n.card-title{font-size:.9rem;font-weight:600;color:var(--accent);margin-bottom:16px;display:flex;align-items:center;gap:8px}\n.panel{display:grid;grid-template-columns:1fr 1fr;gap:18px}\n@media(max-width:720px){.panel{grid-template-columns:1fr}}\n.dropzone{border:2px dashed var(--border);border-radius:var(--radius);background:var(--surface2);min-height:180px;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;cursor:pointer;transition:all .2s;padding:20px;text-align:center}\n.dropzone:hover,.dropzone.drag-over{border-color:var(--accent);background:rgba(88,166,255,.05)}\n.dropzone svg{width:40px;height:40px;opacity:.5}\n.dropzone p{color:var(--muted);font-size:.86rem}\n.dropzone strong{color:var(--text)}\n#file-input{display:none}\n.preview-grid{display:flex;flex-wrap:wrap;gap:8px;margin-top:14px}\n.preview-item{position:relative;width:76px;height:76px;border-radius:8px;overflow:hidden;border:1px solid var(--border)}\n.preview-item img{width:100%;height:100%;object-fit:cover}\n.preview-item .remove{position:absolute;top:2px;right:2px;background:var(--red);border:none;color:#fff;border-radius:50%;width:18px;height:18px;cursor:pointer;font-size:11px;line-height:18px;text-align:center;display:none}\n.preview-item:hover .remove{display:block}\n.preview-item .compress-badge{position:absolute;bottom:2px;left:2px;background:rgba(88,166,255,.85);color:#000;font-size:.6rem;font-weight:700;padding:1px 4px;border-radius:3px}\n.controls{display:flex;flex-direction:column;gap:12px;margin-top:16px}\n.control-row{display:flex;align-items:center;gap:10px;flex-wrap:wrap}\nlabel.toggle{display:flex;align-items:center;gap:8px;cursor:pointer;color:var(--muted);font-size:.86rem}\ninput[type=checkbox]{width:15px;height:15px;accent-color:var(--accent);cursor:pointer}\ninput[type=range]{accent-color:var(--accent);width:110px}\ninput[type=text]{background:var(--surface2);border:1px solid var(--border);border-radius:6px;color:var(--text);padding:5px 10px;font-size:.82rem;width:140px;outline:none}\ninput[type=text]:focus{border-color:var(--accent)}\n.btn{padding:8px 20px;border-radius:8px;border:none;cursor:pointer;font-size:.88rem;font-weight:600;transition:all .18s}\n.btn-primary{background:var(--accent);color:#000}\n.btn-primary:hover:not(:disabled){background:#79b8ff}\n.btn-primary:disabled{opacity:.4;cursor:not-allowed}\n.btn-danger{background:var(--red);color:#fff}\n.btn-danger:hover{background:#da3633}\n.btn-ghost{background:var(--surface2);color:var(--text);border:1px solid var(--border)}\n.btn-ghost:hover{border-color:var(--accent)}\n.btn-sm{padding:5px 14px;font-size:.8rem}\n.output-box{background:var(--surface2);border:1px solid var(--border);border-radius:var(--radius);padding:16px;min-height:160px;max-height:380px;overflow-y:auto;font-family:'Courier New',monospace;font-size:.85rem;line-height:1.85;color:var(--accent2);white-space:pre-wrap;word-break:break-word}\n.output-box.empty{color:var(--muted);font-style:italic;font-family:inherit}\n.output-box.busy{color:var(--warn);font-style:italic;font-family:inherit}\n.output-box.fade-in{animation:fadeIn .4s ease}\n@keyframes fadeIn{from{opacity:.3}to{opacity:1}}\n.output-meta{display:flex;gap:10px;margin-top:8px;flex-wrap:wrap;align-items:center}\n.meta-chip{background:var(--surface2);border:1px solid var(--border);border-radius:20px;padding:2px 10px;font-size:.74rem;color:var(--muted)}\n.meta-chip span{color:var(--text);font-weight:600}\n.export-bar{display:flex;gap:8px;margin-top:8px;flex-wrap:wrap}\n.export-btn{padding:4px 12px;font-size:.74rem;border-radius:6px;background:var(--surface2);border:1px solid var(--border);color:var(--muted);cursor:pointer;transition:all .15s}\n.export-btn:hover{color:var(--text);border-color:var(--accent2)}\n.insight-panel{background:rgba(88,166,255,.06);border:1px solid rgba(88,166,255,.25);border-radius:10px;padding:12px 16px;margin-top:12px;display:none}\n.insight-panel.show{display:block;animation:fadeIn .5s ease}\n.insight-header{font-size:.76rem;font-weight:700;color:var(--accent);text-transform:uppercase;letter-spacing:.06em;margin-bottom:8px;display:flex;align-items:center;gap:6px}\n.insight-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(120px,1fr));gap:8px}\n.insight-chip{background:var(--surface2);border:1px solid var(--border);border-radius:8px;padding:8px 12px;position:relative}\n.insight-chip .ic-label{font-size:.68rem;color:var(--muted);margin-bottom:2px}\n.insight-chip .ic-val{font-size:.84rem;color:var(--text);font-weight:600;word-break:break-all}\n.insight-chip .ic-copy{position:absolute;top:6px;right:6px;background:none;border:none;color:var(--muted);cursor:pointer;font-size:.7rem;opacity:0;transition:opacity .15s}\n.insight-chip:hover .ic-copy{opacity:1}\n.insight-tags{display:flex;flex-wrap:wrap;gap:6px;margin-top:8px}\n.itag{border-radius:20px;padding:2px 10px;font-size:.72rem;font-weight:600}\n.itag-receipt{background:rgba(126,231,135,.15);border:1px solid #7ee787;color:#7ee787}\n.itag-contact{background:rgba(210,168,255,.15);border:1px solid #d2a8ff;color:#d2a8ff}\n.itag-id{background:rgba(121,192,255,.15);border:1px solid #79c0ff;color:#79c0ff}\n.itag-url{background:rgba(240,136,62,.15);border:1px solid #f0883e;color:#f0883e}\n.itag-date{background:rgba(255,210,84,.15);border:1px solid #ffd254;color:#ffd254}\n.itag-number{background:rgba(248,81,73,.15);border:1px solid #f85149;color:#f85149}\n.itag-doc{background:rgba(88,166,255,.15);border:1px solid #58a6ff;color:#58a6ff}\n.batch-result{border-bottom:1px solid var(--border);padding:12px 0}\n.batch-result:last-child{border-bottom:none}\n.batch-result.batch-error{border-left:3px solid var(--red);padding-left:12px}\n.batch-filename{font-size:.82rem;color:var(--accent);font-weight:600;margin-bottom:7px;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px}\n.batch-text{font-family:'Courier New',monospace;font-size:.82rem;color:var(--accent2);white-space:pre-wrap;line-height:1.7}\n.batch-error-msg{color:var(--red);font-size:.82rem;margin-top:4px}\n.copy-btn{padding:4px 12px;font-size:.76rem;border-radius:6px;background:var(--surface2);border:1px solid var(--border);color:var(--muted);cursor:pointer;transition:all .15s}\n.copy-btn:hover{color:var(--text);border-color:var(--accent)}\n.copy-btn.copied{color:var(--accent2);border-color:var(--accent2)}\n.status{display:flex;align-items:center;gap:8px;font-size:.86rem;color:var(--muted);min-height:24px;margin-top:8px}\n.spinner{width:15px;height:15px;border:2px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:spin .8s linear infinite;flex-shrink:0}\n@keyframes spin{to{transform:rotate(360deg)}}\n/* \u2500\u2500 Camera layout \u2500\u2500 */\n.cam-grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}\n@media(max-width:900px){.cam-grid{grid-template-columns:1fr}}\n#camera-video{width:100%;border-radius:10px;border:2px solid var(--border);background:#000;aspect-ratio:4/3;object-fit:cover}\n#camera-canvas{display:none}\n.live-badge{display:inline-flex;align-items:center;gap:6px;background:rgba(240,80,80,.15);border:1px solid var(--red);border-radius:20px;padding:3px 12px;font-size:.74rem;color:var(--red);font-weight:600}\n.live-dot{width:7px;height:7px;background:var(--red);border-radius:50%;animation:blink 1s infinite}\n@keyframes blink{0%,100%{opacity:1}50%{opacity:.2}}\n.cam-controls{display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:12px}\n.frame-info{color:var(--muted);font-size:.78rem}\n.rl-banner{background:rgba(248,81,73,.1);border:1px solid var(--red);border-radius:8px;padding:8px 14px;color:var(--red);font-size:.84rem;display:none;margin-top:8px}\n.rl-banner.show{display:block}\n.section{display:none}\n.section.active{display:flex;flex-direction:column;gap:16px}\n.mode-pills{display:flex;gap:6px}\n.pill{padding:4px 15px;border-radius:20px;border:1px solid var(--border);background:var(--surface2);cursor:pointer;font-size:.82rem;color:var(--muted);transition:all .18s}\n.pill.active{border-color:var(--accent);color:var(--accent);background:rgba(88,166,255,.1)}\n#toast{position:fixed;bottom:22px;left:50%;transform:translateX(-50%);background:var(--accent2);color:#000;padding:8px 22px;border-radius:8px;font-weight:600;opacity:0;transition:opacity .3s;pointer-events:none;z-index:9999}\n#toast.show{opacity:1}\nprogress{width:100%;height:5px;border-radius:3px;margin-top:7px;accent-color:var(--accent)}\n.diff-new{background:rgba(126,231,135,.25);border-radius:3px;padding:0 2px;color:var(--accent2)}\n.cam-state-badge{font-size:.73rem;padding:2px 10px;border-radius:20px;border:1px solid var(--border);background:var(--surface2);color:var(--muted)}\n.cam-state-badge.connecting{border-color:var(--warn);color:var(--warn)}\n.cam-state-badge.active{border-color:var(--accent2);color:var(--accent2)}\n.cam-state-badge.reconnecting{border-color:var(--red);color:var(--red)}\n.motion-dot{width:8px;height:8px;border-radius:50%;background:var(--muted);display:inline-block;margin-right:4px;transition:background .2s}\n.motion-dot.active{background:var(--accent2);box-shadow:0 0 6px var(--accent2)}\n.conf-bar-wrap{display:flex;align-items:center;gap:6px;margin-top:6px}\n.conf-bar-bg{flex:1;height:4px;background:var(--border);border-radius:2px;overflow:hidden}\n.conf-bar-fill{height:100%;border-radius:2px;transition:width .4s,background .4s}\n.conf-label{font-size:.7rem;color:var(--muted);min-width:28px;text-align:right}\n.lock-badge{display:none;font-size:.73rem;padding:2px 10px;border-radius:20px;border:1px solid var(--accent2);background:rgba(126,231,135,.12);color:var(--accent2);font-weight:700}\n.stab-bar{display:inline-block;width:42px;height:5px;background:var(--border);border-radius:3px;margin-left:6px;vertical-align:middle;overflow:hidden}\n.stab-fill{display:block;height:100%;width:0%;background:var(--accent2);border-radius:3px;transition:width .3s}\n/* \u2500\u2500 Accumulated panel (v13 \u2014 with toggle + edit) \u2500\u2500 */\n.acc-panel{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:18px}\n.acc-header{display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;margin-bottom:10px}\n.acc-title{font-size:.92rem;font-weight:600;color:var(--accent2);display:flex;align-items:center;gap:8px}\n.acc-line-count{font-size:.73rem;color:var(--muted)}\n.acc-output{background:var(--surface2);border:1px solid var(--border);border-radius:var(--radius);padding:14px;max-height:480px;overflow-y:auto;font-family:'Courier New',monospace;font-size:.85rem;line-height:1.85;color:var(--text);white-space:pre-wrap;word-break:break-word}\n.acc-output.empty{color:var(--muted);font-style:italic;font-family:inherit}\n.acc-edit-area{width:100%;background:var(--surface2);border:1px solid var(--accent);border-radius:var(--radius);padding:14px;font-family:'Courier New',monospace;font-size:.85rem;line-height:1.85;color:var(--text);min-height:200px;max-height:480px;resize:vertical;outline:none;display:none}\n.acc-disabled-notice{color:var(--muted);font-size:.82rem;font-style:italic;padding:20px 0;text-align:center}\n@keyframes lineIn{from{background:rgba(126,231,135,.22)}to{background:transparent}}\n.line-new{animation:lineIn 1.8s ease forwards;border-radius:2px;display:block}\n/* Toggle switch */\n.toggle-switch{display:flex;align-items:center;gap:8px;font-size:.82rem;color:var(--muted);cursor:pointer;user-select:none}\n.toggle-switch input{display:none}\n.toggle-track{width:36px;height:20px;border-radius:10px;background:var(--border);position:relative;transition:background .2s;flex-shrink:0}\n.toggle-track::after{content:'';position:absolute;left:2px;top:2px;width:16px;height:16px;border-radius:50%;background:#fff;transition:left .2s}\n.toggle-switch input:checked~.toggle-track{background:var(--accent2)}\n.toggle-switch input:checked~.toggle-track::after{left:18px}\n/* history */\n.history-panel{background:var(--surface);border:1px solid var(--border);border-radius:var(--radius);padding:18px}\n.history-list{display:flex;flex-direction:column;gap:10px;max-height:340px;overflow-y:auto;margin-top:12px}\n.history-item{background:var(--surface2);border:1px solid var(--border);border-radius:8px;padding:10px 14px;cursor:pointer;transition:border-color .18s;display:flex;gap:12px;align-items:flex-start}\n.history-item:hover{border-color:var(--accent)}\n.history-thumb{width:46px;height:46px;border-radius:6px;object-fit:cover;flex-shrink:0;background:var(--surface)}\n.history-meta{flex:1;min-width:0}\n.history-name{font-size:.82rem;color:var(--accent);font-weight:600;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\n.history-preview{font-size:.76rem;color:var(--muted);margin-top:3px;overflow:hidden;display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical}\n.history-time{font-size:.7rem;color:var(--muted);margin-top:4px}\n.kbd{display:inline-block;background:var(--surface2);border:1px solid var(--border);border-radius:4px;padding:1px 6px;font-size:.73rem;font-family:monospace;color:var(--muted)}\n.slider-label{font-size:.8rem;color:var(--muted);display:flex;align-items:center;gap:8px}\n.slider-val{color:var(--text);font-weight:600;min-width:28px;text-align:right}\n#offline-banner{position:fixed;top:0;left:0;right:0;background:var(--red);color:#fff;text-align:center;padding:8px;font-weight:600;font-size:.9rem;z-index:10000;display:none}\n</style>\n</head>\n<body>\n<div id=\"offline-banner\">&#9888;&#65039; No internet connection &mdash; OCR paused</div>\n<div class=\"app\">\n  <header>\n    <svg width=\"26\" height=\"26\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#58a6ff\" stroke-width=\"2\">\n      <rect x=\"3\" y=\"3\" width=\"18\" height=\"18\" rx=\"2\"/>\n      <path d=\"M7 8h10M7 12h10M7 16h6\"/>\n    </svg>\n    <h1>OCR Studio</h1>\n    <span style=\"color:var(--muted);font-size:.82rem\">Qwen2-VL-2B &middot; v13</span>\n    <div style=\"flex:1\"></div>\n    <span class=\"badge\" id=\"gpu-badge\">Checking&hellip;</span>\n    <span class=\"badge\" id=\"rl-badge\" style=\"display:none\"></span>\n    <span style=\"color:var(--muted);font-size:.73rem\">\n      <span class=\"kbd\">Ctrl+Enter</span> Run &nbsp;\n      <span class=\"kbd\">Esc</span> Clear &nbsp;\n      <span class=\"kbd\">C</span> Copy\n    </span>\n  </header>\n\n  <main>\n    <div class=\"warn-banner\" id=\"cpu-warn\">\n      &#9888;&#65039; <strong>No GPU detected &mdash; running on CPU.</strong>\n      Each image takes 60&ndash;120 seconds. Go to <strong>Runtime &rarr; Change runtime type &rarr; T4 GPU</strong> then restart all cells.\n    </div>\n\n    <div class=\"tabs\">\n      <div class=\"tab active\" data-tab=\"image\" onclick=\"switchTab('image')\">&#128444;&#65039; Image OCR</div>\n      <div class=\"tab\" data-tab=\"camera\" onclick=\"switchTab('camera')\">&#128247; Live Camera</div>\n      <div class=\"tab\" data-tab=\"history\" onclick=\"switchTab('history')\">&#128336; History</div>\n    </div>\n\n    <!-- IMAGE SECTION -->\n    <div class=\"section active\" id=\"sec-image\">\n      <div style=\"display:flex;align-items:center;gap:12px;flex-wrap:wrap\">\n        <span style=\"color:var(--muted);font-size:.86rem\">Mode:</span>\n        <div class=\"mode-pills\">\n          <div class=\"pill active\" id=\"pill-single\" onclick=\"setImageMode('single')\">Single Image</div>\n          <div class=\"pill\" id=\"pill-batch\" onclick=\"setImageMode('batch')\">Batch (Multiple)</div>\n        </div>\n      </div>\n      <div class=\"panel\">\n        <div class=\"card\">\n          <div class=\"card-title\">\n            <svg width=\"15\" height=\"15\" viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"currentColor\" stroke-width=\"2\">\n              <path d=\"M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4\"/>\n              <polyline points=\"17 8 12 3 7 8\"/><line x1=\"12\" y1=\"3\" x2=\"12\" y2=\"15\"/>\n            </svg>\n            Upload\n          </div>\n          <div class=\"dropzone\" id=\"dropzone\"\n               onclick=\"document.getElementById('file-input').click()\"\n               ondragover=\"onDragOver(event)\" ondragleave=\"onDragLeave(event)\" ondrop=\"onDrop(event)\">\n            <svg viewBox=\"0 0 24 24\" fill=\"none\" stroke=\"#58a6ff\" stroke-width=\"1.5\">\n              <rect x=\"3\" y=\"3\" width=\"18\" height=\"18\" rx=\"2\"/>\n              <path d=\"M8 12l4-4 4 4M12 8v8\"/>\n            </svg>\n            <p><strong>Click to upload</strong> or drag &amp; drop</p>\n            <p>PNG, JPG, JPEG, WEBP &mdash; <span id=\"multi-hint\">one file</span></p>\n            <p style=\"font-size:.74rem;color:var(--accent);margin-top:4px\">&#9889; Auto-compressed to 1024px</p>\n          </div>\n          <input type=\"file\" id=\"file-input\" accept=\"image/*\" onchange=\"onFileChange(event)\"/>\n          <div class=\"preview-grid\" id=\"preview-grid\"></div>\n          <div class=\"controls\">\n            <div class=\"control-row\">\n              <label class=\"toggle\"><input type=\"checkbox\" id=\"auto-crop\" checked/>Auto-crop document</label>\n            </div>\n            <div class=\"control-row\">\n              <label class=\"slider-label\">Min score:\n                <input type=\"range\" id=\"min-score\" min=\"-200\" max=\"200\" value=\"-200\" oninput=\"document.getElementById('score-val').textContent=this.value<-199?'Off':this.value\"/>\n                <span class=\"slider-val\" id=\"score-val\">Off</span>\n              </label>\n            </div>\n            <div class=\"control-row\">\n              <label style=\"font-size:.82rem;color:var(--muted)\">Language hint:</label>\n              <input type=\"text\" id=\"lang-hint\" placeholder=\"e.g. Hindi, Arabic&hellip;\"/>\n            </div>\n            <div class=\"control-row\">\n              <button class=\"btn btn-primary\" id=\"run-btn\" onclick=\"runOCR()\" disabled>&#128269; Run OCR</button>\n              <button class=\"btn btn-ghost\" onclick=\"clearFiles()\">Clear</button>\n            </div>\n          </div>\n          <div class=\"status\" id=\"img-status\"></div>\n          <div class=\"rl-banner\" id=\"img-rl-banner\"></div>\n          <progress id=\"img-progress\" style=\"display:none\" max=\"100\"></progress>\n        </div>\n        <div class=\"card\">\n          <div class=\"card-title\" style=\"justify-content:space-between\">\n            <span>&#128196; Extracted Text</span>\n            <button class=\"copy-btn\" onclick=\"copyOutput()\">Copy</button>\n          </div>\n          <div id=\"single-output\">\n            <div class=\"output-box empty\" id=\"output-text\">Extracted text will appear here&hellip;</div>\n            <div class=\"output-meta\" id=\"output-meta\"></div>\n            <div class=\"export-bar\" id=\"img-export-bar\" style=\"display:none\">\n              <button class=\"export-btn\" onclick=\"exportResult('txt')\">&#8681; .txt</button>\n              <button class=\"export-btn\" onclick=\"exportResult('json')\">&#8681; .json</button>\n            </div>\n            <div class=\"insight-panel\" id=\"img-insight\"></div>\n          </div>\n          <div id=\"batch-output\" style=\"display:none\">\n            <div id=\"batch-results\"></div>\n            <div class=\"export-bar\" id=\"batch-export-bar\" style=\"display:none\">\n              <button class=\"export-btn\" onclick=\"exportBatch('txt')\">&#8681; All .txt</button>\n              <button class=\"export-btn\" onclick=\"exportBatch('csv')\">&#8681; .csv</button>\n              <button class=\"export-btn\" onclick=\"exportBatch('json')\">&#8681; .json</button>\n            </div>\n          </div>\n        </div>\n      </div>\n    </div>\n\n    <!-- CAMERA SECTION -->\n    <div class=\"section\" id=\"sec-camera\">\n      <div class=\"card\">\n        <!-- Controls row -->\n        <div class=\"cam-controls\">\n          <button class=\"btn btn-primary\" id=\"cam-start-btn\" onclick=\"startCamera()\">&#128247; Start Camera</button>\n          <button class=\"btn btn-danger\" id=\"cam-stop-btn\" onclick=\"stopCamera()\" style=\"display:none\">&#9209; Stop</button>\n          <span class=\"live-badge\" id=\"live-badge\" style=\"display:none\">\n            <span class=\"live-dot\"></span> LIVE\n            <span class=\"stab-bar\"><span class=\"stab-fill\" id=\"stab-fill\"></span></span>\n          </span>\n          <span class=\"lock-badge\" id=\"lock-badge\">&#128274; LOCKED</span>\n          <span class=\"cam-state-badge\" id=\"cam-state-badge\">IDLE</span>\n          <span class=\"frame-info\" id=\"frame-info\"></span>\n        </div>\n\n        <!-- Two-panel layout: video left, current OCR right -->\n        <div class=\"cam-grid\">\n          <div>\n            <div class=\"card-title\">\n              <span class=\"motion-dot\" id=\"motion-dot\"></span>&#128247; Camera Feed\n            </div>\n            <video id=\"camera-video\" autoplay muted playsinline></video>\n            <canvas id=\"camera-canvas\"></canvas>\n          </div>\n          <div>\n            <div class=\"card-title\" style=\"justify-content:space-between\">\n              <span>&#128221; Current Frame</span>\n              <div style=\"display:flex;gap:6px\">\n                <button class=\"copy-btn btn-sm\" onclick=\"copyCameraOutput()\">Copy</button>\n                <button class=\"export-btn\" onclick=\"exportCameraResult()\" style=\"font-size:.73rem\">&#8681; .txt</button>\n              </div>\n            </div>\n            <div class=\"output-box empty\" id=\"camera-output\" style=\"min-height:200px;font-family:inherit;font-style:italic\">\n              Start the camera to begin live OCR&hellip;\n            </div>\n            <div class=\"conf-bar-wrap\" id=\"conf-bar-wrap\" style=\"display:none\">\n              <span style=\"font-size:.7rem;color:var(--muted)\">Confidence</span>\n              <div class=\"conf-bar-bg\"><div class=\"conf-bar-fill\" id=\"conf-bar-fill\"></div></div>\n              <span class=\"conf-label\" id=\"conf-label\">-</span>\n            </div>\n            <div class=\"output-meta\" id=\"camera-meta\"></div>\n            <div class=\"insight-panel\" id=\"cam-insight\"></div>\n          </div>\n        </div>\n\n        <div class=\"status\" id=\"cam-status\" style=\"margin-top:12px\"></div>\n        <div class=\"rl-banner\" id=\"cam-rl-banner\"></div>\n      </div>\n\n      <!-- v13: Accumulated Scan \u2014 with toggle + edit -->\n      <div class=\"acc-panel\" id=\"acc-panel\">\n        <div class=\"acc-header\">\n          <div class=\"acc-title\">\n            &#128230; Accumulated Scan\n            <span class=\"acc-line-count\" id=\"acc-line-count\"></span>\n          </div>\n          <div style=\"display:flex;gap:8px;flex-wrap:wrap;align-items:center\">\n            <!-- Enable/Disable toggle -->\n            <label class=\"toggle-switch\" title=\"Enable or disable accumulation of scan results\">\n              <input type=\"checkbox\" id=\"acc-toggle\" checked onchange=\"onAccToggle()\"/>\n              <span class=\"toggle-track\"></span>\n              <span id=\"acc-toggle-label\">Accumulate: ON</span>\n            </label>\n            <button class=\"copy-btn\" id=\"acc-copy-btn\" onclick=\"copyAccOutput()\">Copy</button>\n            <button class=\"export-btn\" id=\"acc-export-txt\" onclick=\"exportAccResult('txt')\">&#8681; .txt</button>\n            <button class=\"export-btn\" id=\"acc-export-json\" onclick=\"exportAccResult('json')\">&#8681; .json</button>\n            <!-- Edit / Save / Cancel buttons -->\n            <button class=\"btn btn-ghost btn-sm\" id=\"acc-edit-btn\" onclick=\"startEditAcc()\">&#9998; Edit</button>\n            <button class=\"btn btn-primary btn-sm\" id=\"acc-save-btn\" onclick=\"saveEditAcc()\" style=\"display:none\">&#10003; Save</button>\n            <button class=\"btn btn-ghost btn-sm\" id=\"acc-cancel-btn\" onclick=\"cancelEditAcc()\" style=\"display:none\">&#10005; Cancel</button>\n            <button class=\"btn btn-ghost btn-sm\" id=\"acc-clear-btn\" onclick=\"clearAccumulated()\">&#128465; Clear</button>\n          </div>\n        </div>\n        <!-- View mode -->\n        <div class=\"acc-output empty\" id=\"acc-output\">\n          Scan results appear here. New unique lines are appended as you scan different parts of the document.\n        </div>\n        <!-- Disabled notice (shown when toggle is OFF) -->\n        <div class=\"acc-disabled-notice\" id=\"acc-disabled-notice\" style=\"display:none\">\n          &#128526; Accumulation is <strong>OFF</strong> &mdash; only current frame is shown.\n        </div>\n        <!-- Edit textarea (shown during editing) -->\n        <textarea class=\"acc-edit-area\" id=\"acc-edit-area\" placeholder=\"Edit accumulated text here&hellip;\"></textarea>\n      </div>\n    </div>\n\n    <!-- HISTORY SECTION -->\n    <div class=\"section\" id=\"sec-history\">\n      <div class=\"history-panel\">\n        <div class=\"card-title\" style=\"justify-content:space-between\">\n          <span>&#128336; Recent OCR Results <span style=\"color:var(--muted);font-weight:400;font-size:.78rem\">(session only)</span></span>\n          <button class=\"btn btn-ghost btn-sm\" onclick=\"clearHistory()\">Clear All</button>\n        </div>\n        <div class=\"history-list\" id=\"history-list\">\n          <div style=\"color:var(--muted);font-size:.86rem;padding:12px 0\">No history yet.</div>\n        </div>\n      </div>\n    </div>\n  </main>\n</div>\n<div id=\"toast\">Copied!</div>\n\n<script>\n// \u2500\u2500\u2500 CONFIG \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nconst CONFIG = {\n  WS_RECONNECT_DELAYS:  [1000,2000,4000,8000,16000],\n  FRAME_SEND_DELAY:     50,\n  FRAME_BUSY_RETRY:     900,\n  FRAME_RL_RETRY:       1200,\n  FRAME_SAFETY_TIMEOUT: 30000,\n  COMPRESS_MAX_DIM:     1024,\n  COMPRESS_QUALITY:     0.88,\n  CAM_JPEG_QUALITY:     0.88,\n  HISTORY_MAX:          10,\n  STABLE_LOCK_THRESHOLD:4,\n  MOTION_THRESHOLD:     0.022,\n  HIGH_CONF_SCORE:      50,\n  HIGH_CONF_FRAMES_REQ: 3,\n  ADAPTIVE_SAME_LIMIT:  5,\n  ADAPTIVE_SLOW_DELAY:  2800,\n  MIN_FRAME_SCORE:      25,\n};\n\n// \u2500\u2500\u2500 CAMERA STATE \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nconst CAM_STATE={IDLE:'IDLE',CONNECTING:'CONNECTING',ACTIVE:'ACTIVE',RECONNECTING:'RECONNECTING',STOPPED:'STOPPED'};\nlet camState=CAM_STATE.IDLE;\nfunction setCamState(s){\n  camState=s;\n  const b=document.getElementById('cam-state-badge');\n  b.textContent=s;\n  b.className='cam-state-badge'+(s==='ACTIVE'?' active':s==='RECONNECTING'?' reconnecting':s==='CONNECTING'?' connecting':'');\n}\n\n// \u2500\u2500\u2500 GLOBALS \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nconst API=window.location.origin;\nlet imageMode='single',files=[],compressedFiles=[];\nlet cameraStream=null,cameraWS=null;\nlet frameCount=0,lastCamText='',pendingFrame=false;\nlet elapsedInterval=null,elapsedStart=0,progressAnim=null;\nlet wsReconnectAttempt=0,wsReconnectTimer=null;\nlet lastFrameHash='',sameTextCount=0;\nlet batchData=null,lastSingleResult=null;\nlet lastFramePixels=null;\nlet currentScore=0,highConfFrames=0,lockedOnText=false;\n// v13: accumulated text comes from the SERVER; toggle + edit state\nlet serverAccumulated='';\nlet accumulateEnabled=true;  // driven by toggle\nlet accEditMode=false;       // true while textarea is open\n\n// \u2500\u2500\u2500 OFFLINE \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nwindow.addEventListener('offline',()=>{\n  document.getElementById('offline-banner').style.display='block';\n  if(camState===CAM_STATE.ACTIVE){setCamState(CAM_STATE.RECONNECTING);setStatus('cam-status',false,'&#9888;&#65039; Offline \u2014 paused');}\n});\nwindow.addEventListener('online',()=>{\n  document.getElementById('offline-banner').style.display='none';\n  if(camState===CAM_STATE.RECONNECTING&&cameraStream)connectCameraWS();\n});\n\n// \u2500\u2500\u2500 KEYBOARD \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\ndocument.addEventListener('keydown',e=>{\n  const tag=document.activeElement.tagName;\n  if(tag==='INPUT'||tag==='TEXTAREA')return;\n  if(e.ctrlKey&&e.key==='Enter'){e.preventDefault();runOCR();return;}\n  if(e.key==='Escape'){clearFiles();return;}\n  if(e.key==='c'&&!e.ctrlKey){copyOutput();return;}\n});\n\n// \u2500\u2500\u2500 TABS \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction switchTab(tab){\n  document.querySelectorAll('.tab').forEach(t=>t.classList.toggle('active',t.dataset.tab===tab));\n  document.querySelectorAll('.section').forEach(s=>s.classList.remove('active'));\n  document.getElementById('sec-'+tab).classList.add('active');\n  if(tab==='history')renderHistory();\n}\n\n// \u2500\u2500\u2500 IMAGE MODE \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction setImageMode(mode){\n  imageMode=mode;\n  document.getElementById('pill-single').classList.toggle('active',mode==='single');\n  document.getElementById('pill-batch').classList.toggle('active',mode==='batch');\n  document.getElementById('file-input').multiple=mode==='batch';\n  document.getElementById('multi-hint').textContent=mode==='batch'?'multiple files':'one file';\n  clearFiles();\n}\n\n// \u2500\u2500\u2500 DRAG & DROP \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction onDragOver(e){e.preventDefault();document.getElementById('dropzone').classList.add('drag-over')}\nfunction onDragLeave(){document.getElementById('dropzone').classList.remove('drag-over')}\nfunction onDrop(e){\n  e.preventDefault();document.getElementById('dropzone').classList.remove('drag-over');\n  const dropped=Array.from(e.dataTransfer.files).filter(f=>f.type.startsWith('image/'));\n  imageMode==='single'?addFiles([dropped[0]].filter(Boolean)):addFiles(dropped);\n}\nfunction onFileChange(e){\n  const sel=Array.from(e.target.files);\n  imageMode==='single'?addFiles([sel[0]].filter(Boolean)):addFiles(sel);\n  e.target.value='';\n}\n\n// \u2500\u2500\u2500 COMPRESSION \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nasync function compressImage(file){\n  return new Promise(resolve=>{\n    const img=new Image(),url=URL.createObjectURL(file);\n    img.onload=()=>{\n      URL.revokeObjectURL(url);\n      const max=CONFIG.COMPRESS_MAX_DIM,w=img.naturalWidth,h=img.naturalHeight;\n      if(Math.max(w,h)<=max){resolve({file,compressed:false});return;}\n      const scale=max/Math.max(w,h),nw=Math.round(w*scale),nh=Math.round(h*scale);\n      const canvas=document.createElement('canvas');\n      canvas.width=nw;canvas.height=nh;\n      canvas.getContext('2d').drawImage(img,0,0,nw,nh);\n      canvas.toBlob(blob=>{\n        resolve({file:new File([blob],file.name,{type:'image/jpeg'}),compressed:true,origW:w,origH:h});\n      },'image/jpeg',CONFIG.COMPRESS_QUALITY);\n    };\n    img.onerror=()=>resolve({file,compressed:false});\n    img.src=url;\n  });\n}\nasync function addFiles(newFiles){\n  files=imageMode==='single'?newFiles.slice(0,1):[...files,...newFiles];\n  compressedFiles=await Promise.all(files.map(f=>compressImage(f)));\n  renderPreviews();\n  document.getElementById('run-btn').disabled=files.length===0;\n}\nfunction removeFile(idx){files.splice(idx,1);compressedFiles.splice(idx,1);renderPreviews();document.getElementById('run-btn').disabled=files.length===0;}\nfunction clearFiles(){files=[];compressedFiles=[];renderPreviews();document.getElementById('run-btn').disabled=true;resetOutput();}\nfunction renderPreviews(){\n  const g=document.getElementById('preview-grid');g.innerHTML='';\n  files.forEach((f,i)=>{\n    const url=URL.createObjectURL(f),cInfo=compressedFiles[i];\n    g.innerHTML+=`<div class=\"preview-item\"><img src=\"${url}\"/>\n      <button class=\"remove\" onclick=\"removeFile(${i})\">&#x2715;</button>\n      ${cInfo&&cInfo.compressed?'<span class=\"compress-badge\">&#9889;</span>':''}\n    </div>`;\n  });\n}\n\n// \u2500\u2500\u2500 PROGRESS \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction startElapsed(sid){\n  stopElapsed();elapsedStart=Date.now();\n  elapsedInterval=setInterval(()=>{\n    const s=((Date.now()-elapsedStart)/1000).toFixed(0),el=document.getElementById(sid);\n    if(el){const t=el.querySelector('.elapsed-txt');if(t)t.textContent=s+'s';}\n  },1000);\n}\nfunction stopElapsed(){if(elapsedInterval){clearInterval(elapsedInterval);elapsedInterval=null;}}\nfunction animateProgress(){\n  const p=document.getElementById('img-progress');let v=0;p.value=0;\n  if(progressAnim)clearInterval(progressAnim);\n  progressAnim=setInterval(()=>{v=Math.min(v+0.5,90);p.value=v;},300);\n}\nfunction finishProgress(){\n  if(progressAnim){clearInterval(progressAnim);progressAnim=null;}\n  const p=document.getElementById('img-progress');if(p)p.value=100;\n}\n\n// \u2500\u2500\u2500 IMAGE OCR \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nasync function runOCR(){\n  if(!files.length)return;\n  const autoCrop=document.getElementById('auto-crop').checked;\n  document.getElementById('run-btn').disabled=true;\n  document.getElementById('img-progress').style.display='';\n  document.getElementById('img-rl-banner').classList.remove('show');\n  animateProgress();\n  try{\n    const cFiles=compressedFiles.map(c=>c.file);\n    if(imageMode==='single'||files.length===1)await runSingleOCR(cFiles[0]||files[0],autoCrop);\n    else await runBatchOCR(cFiles.length?cFiles:files,autoCrop);\n  }catch(err){setStatus('img-status',false,'&#10060; '+err.message);}\n  finally{document.getElementById('run-btn').disabled=false;document.getElementById('img-progress').style.display='none';stopElapsed();}\n}\nfunction getLangHint(){const v=document.getElementById('lang-hint').value.trim();return v?` The text is in ${v}.`:''}\nfunction getMinScore(){const v=parseInt(document.getElementById('min-score').value);return v<=-199?-Infinity:v;}\nasync function runSingleOCR(file,autoCrop){\n  const fd=new FormData();fd.append('file',file);fd.append('auto_crop',autoCrop);\n  const hint=getLangHint();if(hint)fd.append('lang_hint',hint);\n  setStatus('img-status',true,`Processing <b>${file.name}</b>&hellip; <span class=\"elapsed-txt\">0s</span>`);\n  startElapsed('img-status');\n  const res=await fetch(`${API}/ocr/image`,{method:'POST',body:fd});\n  if(res.status===429){const d=await res.json();document.getElementById('img-rl-banner').textContent='&#9888;&#65039; '+d.error;document.getElementById('img-rl-banner').classList.add('show');finishProgress();stopElapsed();setStatus('img-status',false,'');return;}\n  const data=await res.json();\n  finishProgress();stopElapsed();\n  if(data.error)throw new Error(data.error);\n  const minScore=getMinScore();\n  if(data.score<minScore){setStatus('img-status',false,`&#9888;&#65039; Low confidence (score ${data.score} < ${minScore}) \u2014 result hidden`);return;}\n  lastSingleResult=data;\n  document.getElementById('single-output').style.display='';\n  document.getElementById('batch-output').style.display='none';\n  const box=document.getElementById('output-text');\n  box.className='output-box fade-in';box.style.color='var(--accent2)';box.style.fontStyle='normal';box.style.fontFamily=\"'Courier New',monospace\";box.textContent=data.text;\n  document.getElementById('output-meta').innerHTML=\n    `<div class=\"meta-chip\">&#9201; <span>${data.elapsed}s</span></div>`+\n    `<div class=\"meta-chip\">&#128202; Score <span>${data.score}</span></div>`+\n    `<div class=\"meta-chip\">&#128421; <span>${data.device||'?'}</span></div>`+\n    (data.angle_corrected?`<div class=\"meta-chip\">&#128260; <span>${data.angle_corrected}&deg;</span></div>`:'');\n  document.getElementById('img-export-bar').style.display='';\n  renderInsight('img-insight',data.text);\n  setStatus('img-status',false,`&#9989; Done in ${data.elapsed}s`);\n  saveToHistory(file.name,data.text,data.elapsed,URL.createObjectURL(file));\n}\nasync function runBatchOCR(fileList,autoCrop){\n  const fd=new FormData();\n  fileList.forEach(f=>fd.append('files',f));\n  fd.append('auto_crop',autoCrop);\n  const hint=getLangHint();if(hint)fd.append('lang_hint',hint);\n  setStatus('img-status',true,`Processing ${fileList.length} images&hellip; <span class=\"elapsed-txt\">0s</span>`);\n  startElapsed('img-status');\n  const res=await fetch(`${API}/ocr/batch`,{method:'POST',body:fd});\n  if(res.status===429){const d=await res.json();document.getElementById('img-rl-banner').textContent='&#9888;&#65039; '+d.error;document.getElementById('img-rl-banner').classList.add('show');finishProgress();stopElapsed();setStatus('img-status',false,'');return;}\n  const data=await res.json();\n  finishProgress();stopElapsed();\n  batchData=data;\n  const minScore=getMinScore();\n  document.getElementById('single-output').style.display='none';\n  document.getElementById('batch-output').style.display='';\n  const container=document.getElementById('batch-results');container.innerHTML='';\n  let ok=0,fail=0;\n  data.results.forEach(r=>{\n    if(r.error){fail++;container.innerHTML+=`<div class=\"batch-result batch-error\"><div class=\"batch-filename\"><span>&#128196; ${esc(r.filename)}</span><span class=\"badge err\">Error</span></div><div class=\"batch-error-msg\">&#10060; ${esc(r.error)}</div></div>`;return;}\n    if(r.score<minScore){container.innerHTML+=`<div class=\"batch-result\"><div class=\"batch-filename\"><span>&#128196; ${esc(r.filename)}</span><span class=\"badge warn\">Low confidence (${r.score})</span></div></div>`;return;}\n    ok++;\n    container.innerHTML+=`<div class=\"batch-result\"><div class=\"batch-filename\"><span>&#128196; ${esc(r.filename)}</span><div style=\"display:flex;gap:6px;align-items:center\"><span class=\"meta-chip\">&#9201; <span>${r.elapsed}s</span></span><span class=\"meta-chip\">Score <span>${r.score}</span></span><button class=\"copy-btn\" onclick=\"copyText(${JSON.stringify(r.text)},this)\">Copy</button></div></div><div class=\"batch-text\">${esc(r.text)}</div></div>`;\n  });\n  document.getElementById('batch-export-bar').style.display='';\n  setStatus('img-status',false,`&#9989; ${ok} done${fail?`, ${fail} failed`:''}`);\n}\nfunction resetOutput(){\n  lastSingleResult=null;batchData=null;\n  document.getElementById('single-output').style.display='';\n  document.getElementById('batch-output').style.display='none';\n  const box=document.getElementById('output-text');\n  box.className='output-box empty';box.textContent='Extracted text will appear here\\u2026';\n  document.getElementById('output-meta').innerHTML='';\n  document.getElementById('img-insight').classList.remove('show');\n  document.getElementById('img-export-bar').style.display='none';\n  document.getElementById('batch-export-bar').style.display='none';\n}\n\n// \u2500\u2500\u2500 EXPORT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction downloadFile(name,content,mime){const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([content],{type:mime}));a.download=name;a.click();}\nfunction exportResult(fmt){\n  if(!lastSingleResult)return;\n  const d=lastSingleResult;\n  if(fmt==='txt')downloadFile('ocr_result.txt',d.text,'text/plain');\n  else if(fmt==='json')downloadFile('ocr_result.json',JSON.stringify(d,null,2),'application/json');\n}\nfunction exportBatch(fmt){\n  if(!batchData)return;\n  if(fmt==='txt'){downloadFile('batch_ocr.txt',batchData.results.map(r=>`=== ${r.filename} ===\\n${r.text||r.error||''}`).join('\\n\\n'),'text/plain');}\n  else if(fmt==='csv'){const rows=[['filename','text','elapsed','score'],...batchData.results.map(r=>[`\"${(r.filename||'').replace(/\"/g,'\"\"')}\"`,`\"${(r.text||r.error||'').replace(/\"/g,'\"\"')}\"`,r.elapsed||'',r.score||''])];downloadFile('batch_ocr.csv',rows.map(r=>r.join(',')).join('\\n'),'text/csv');}\n  else if(fmt==='json'){downloadFile('batch_ocr.json',JSON.stringify(batchData,null,2),'application/json');}\n}\nfunction exportCameraResult(){\n  const text=document.getElementById('camera-output').textContent;\n  if(!text||text.includes('Start the camera'))return;\n  downloadFile('frame_ocr.txt',text,'text/plain');\n}\nfunction exportAccResult(fmt){\n  const raw=serverAccumulated||document.getElementById('acc-output').textContent||'';\n  const clean=raw.split('\\n').filter(l=>l.trim()&&l.trim()!=='[No text found]'&&l.trim()!=='[No text detected]').join('\\n');\n  if(!clean)return;\n  if(fmt==='txt')downloadFile('accumulated_ocr.txt',clean,'text/plain');\n  else if(fmt==='json')downloadFile('accumulated_ocr.json',JSON.stringify({accumulated:clean},null,2),'application/json');\n}\n\n// \u2500\u2500\u2500 ACCUMULATED SCAN TOGGLE \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction onAccToggle(){\n  accumulateEnabled=document.getElementById('acc-toggle').checked;\n  document.getElementById('acc-toggle-label').textContent=\n    'Accumulate: '+(accumulateEnabled?'ON':'OFF');\n  const notice=document.getElementById('acc-disabled-notice');\n  const output=document.getElementById('acc-output');\n  if(!accumulateEnabled){\n    // Cancel any edit in progress when disabling\n    if(accEditMode)cancelEditAcc();\n    output.style.display='none';\n    notice.style.display='';\n    // Dim action buttons\n    document.getElementById('acc-edit-btn').disabled=true;\n    document.getElementById('acc-copy-btn').disabled=true;\n    document.getElementById('acc-export-txt').disabled=true;\n    document.getElementById('acc-export-json').disabled=true;\n    document.getElementById('acc-clear-btn').disabled=true;\n  } else {\n    output.style.display='';\n    notice.style.display='none';\n    document.getElementById('acc-edit-btn').disabled=false;\n    document.getElementById('acc-copy-btn').disabled=false;\n    document.getElementById('acc-export-txt').disabled=false;\n    document.getElementById('acc-export-json').disabled=false;\n    document.getElementById('acc-clear-btn').disabled=false;\n  }\n}\n\n// \u2500\u2500\u2500 ACCUMULATED SCAN EDIT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction startEditAcc(){\n  if(accEditMode)return;\n  accEditMode=true;\n  const output=document.getElementById('acc-output');\n  const editArea=document.getElementById('acc-edit-area');\n  const editBtn=document.getElementById('acc-edit-btn');\n  const saveBtn=document.getElementById('acc-save-btn');\n  const cancelBtn=document.getElementById('acc-cancel-btn');\n  // Sync serverAccumulated from DOM in case WS was never open or variable was lost\n  if(!serverAccumulated&&output&&!output.classList.contains('empty')){\n    serverAccumulated=output.innerText||output.textContent||'';\n  }\n  // Filter out [No text found] noise lines before editing\n  const cleanedForEdit=serverAccumulated.split('\\n')\n    .filter(l=>l.trim()&&l.trim()!=='[No text found]'&&l.trim()!=='[No text detected]')\n    .join('\\n');\n  editArea.value=cleanedForEdit;\n  output.style.display='none';\n  editArea.style.display='';\n  editArea.focus();\n  editBtn.style.display='none';\n  saveBtn.style.display='';\n  cancelBtn.style.display='';\n}\n\nfunction saveEditAcc(){\n  if(!accEditMode)return;\n  const editArea=document.getElementById('acc-edit-area');\n  const newText=editArea.value;\n  // Send to server so it uses the edited version going forward\n  if(cameraWS&&cameraWS.readyState===WebSocket.OPEN){\n    cameraWS.send(JSON.stringify({command:'set_accumulated',text:newText}));\n  }\n  // Optimistically update local state\n  serverAccumulated=newText;\n  _exitEditMode();\n  renderAccumulated(newText,undefined);\n  showToast('Accumulated scan updated');\n}\n\nfunction cancelEditAcc(){\n  _exitEditMode();\n}\n\nfunction _exitEditMode(){\n  accEditMode=false;\n  const output=document.getElementById('acc-output');\n  const editArea=document.getElementById('acc-edit-area');\n  const editBtn=document.getElementById('acc-edit-btn');\n  const saveBtn=document.getElementById('acc-save-btn');\n  const cancelBtn=document.getElementById('acc-cancel-btn');\n  editArea.style.display='none';\n  output.style.display='';\n  editBtn.style.display='';\n  saveBtn.style.display='none';\n  cancelBtn.style.display='none';\n}\n\n// \u2500\u2500\u2500 ACCUMULATED SCAN RENDER (v13 \u2014 SERVER-DRIVEN) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction renderAccumulated(newText,prevText){\n  const box=document.getElementById('acc-output');\n  const count=document.getElementById('acc-line-count');\n  if(!accumulateEnabled)return;  // Don't update display when disabled\n  if(accEditMode)return;          // Don't overwrite while user is editing\n  if(!newText||!newText.trim()){\n    box.className='acc-output empty';\n    box.textContent='Scan results appear here. New unique lines are appended as you scan different parts of the document.';\n    count.textContent='';\n    return;\n  }\n  const lines=newText.split('\\n').filter(l=>l.trim());\n  count.textContent=`(${lines.length} line${lines.length!==1?'s':''})`;\n  box.className='acc-output';\n  const prevLines=new Set((prevText||'').split('\\n').map(l=>l.trim()).filter(Boolean));\n  let html='';\n  for(const line of lines){\n    const escaped=esc(line);\n    html+=(!prevLines.has(line.trim())&&prevText!==undefined)\n      ?`<span class=\"line-new\">${escaped}</span>\\n`\n      :escaped+'\\n';\n  }\n  box.innerHTML=html;\n  box.scrollTop=box.scrollHeight;\n  // Keep serverAccumulated in sync with what we just rendered\n  serverAccumulated=newText;\n}\n\nfunction clearAccumulated(){\n  if(cameraWS&&cameraWS.readyState===WebSocket.OPEN){\n    cameraWS.send(JSON.stringify({command:'clear'}));\n  }\n  serverAccumulated='';\n  renderAccumulated('',undefined);\n  showToast('Accumulated scan cleared');\n}\n\nfunction copyAccOutput(){\n  const text=(serverAccumulated||document.getElementById('acc-output').textContent||'')\n    .split('\\n').filter(l=>l.trim()&&l.trim()!=='[No text found]'&&l.trim()!=='[No text detected]').join('\\n');\n  copyText(text);\n}\n\n// \u2500\u2500\u2500 MOTION DETECTION \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction checkMotion(canvas){\n  const sw=24,sh=18;\n  const tmp=document.createElement('canvas');tmp.width=sw;tmp.height=sh;\n  tmp.getContext('2d').drawImage(canvas,0,0,sw,sh);\n  const px=tmp.getContext('2d').getImageData(0,0,sw,sh).data;\n  if(!lastFramePixels){lastFramePixels=new Uint8ClampedArray(px);return true;}\n  let diff=0;\n  for(let i=0;i<px.length;i+=4){diff+=Math.abs(px[i]-lastFramePixels[i])+Math.abs(px[i+1]-lastFramePixels[i+1])+Math.abs(px[i+2]-lastFramePixels[i+2]);}\n  lastFramePixels=new Uint8ClampedArray(px);\n  const ratio=diff/(px.length/4*255*3);\n  document.getElementById('motion-dot').classList.toggle('active',ratio>CONFIG.MOTION_THRESHOLD);\n  return ratio>CONFIG.MOTION_THRESHOLD;\n}\n\n// \u2500\u2500\u2500 CONFIDENCE METER \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction updateConfMeter(score){\n  const pct=Math.max(0,Math.min(100,((score+50)/200)*100));\n  const fill=document.getElementById('conf-bar-fill');\n  const label=document.getElementById('conf-label');\n  document.getElementById('conf-bar-wrap').style.display='flex';\n  fill.style.width=pct+'%';\n  fill.style.background=pct>65?'var(--accent2)':pct>35?'var(--warn)':'var(--red)';\n  label.textContent=score;\n}\n\n// \u2500\u2500\u2500 CAMERA \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nasync function startCamera(){\n  try{\n    cameraStream=await navigator.mediaDevices.getUserMedia({\n      video:{facingMode:{ideal:'environment'},width:{ideal:1920},height:{ideal:1080}}\n    });\n    const video=document.getElementById('camera-video');\n    video.srcObject=cameraStream;await video.play();\n    document.getElementById('cam-start-btn').style.display='none';\n    document.getElementById('cam-stop-btn').style.display='';\n    document.getElementById('live-badge').style.display='';\n    wsReconnectAttempt=0;\n    setCamState(CAM_STATE.CONNECTING);\n    setStatus('cam-status',true,'Connecting to OCR server\\u2026');\n    connectCameraWS();\n  }catch(err){setStatus('cam-status',false,'&#10060; Camera error: '+err.message);}\n}\n\nfunction frameHash(b64){let h=0;for(let i=0;i<b64.length;i+=200)h=(Math.imul(31,h)+b64.charCodeAt(i))|0;return h.toString(36);}\n\nfunction buildDiffHTML(oldTxt,newTxt){\n  const oldSet=new Set(oldTxt.split(/\\s+/));\n  return newTxt.split('\\n').map(line=>line.split(/\\s+/).map(w=>w?(oldSet.has(w)?esc(w):`<span class=\"diff-new\">${esc(w)}</span>`):'').join(' ')).join('\\n');\n}\n\nfunction connectCameraWS(){\n  if(wsReconnectTimer){clearTimeout(wsReconnectTimer);wsReconnectTimer=null;}\n  const wsURL=`${API.replace('http','ws')}/ocr/camera`;\n  cameraWS=new WebSocket(wsURL);\n\n  cameraWS.onopen=()=>{\n    wsReconnectAttempt=0;\n    setCamState(CAM_STATE.ACTIVE);\n    setStatus('cam-status',false,'&#9989; Connected \\u2014 live OCR starting\\u2026');\n    pendingFrame=false;sameTextCount=0;lastFrameHash='';lockedOnText=false;highConfFrames=0;\n    sendFrame();\n  };\n\n  cameraWS.onmessage=(e)=>{\n    const data=JSON.parse(e.data);\n    if(data.heartbeat)return;\n    if(data.error){setStatus('cam-status',false,'&#9888;&#65039; '+data.error);pendingFrame=false;return;}\n    if(data.command_ack==='cleared'){serverAccumulated='';renderAccumulated('',undefined);return;}\n    if(data.command_ack==='set_accumulated'){\n      serverAccumulated=data.accumulated_text||'';\n      renderAccumulated(serverAccumulated,undefined);\n      return;\n    }\n    if(data.rate_limited){\n      document.getElementById('cam-rl-banner').textContent='&#9201; '+data.current_text;\n      document.getElementById('cam-rl-banner').classList.add('show');\n      pendingFrame=false;setTimeout(sendFrame,CONFIG.FRAME_RL_RETRY);return;\n    }\n    document.getElementById('cam-rl-banner').classList.remove('show');\n\n    const box=document.getElementById('camera-output');\n    if(data.busy){\n      box.className='output-box busy';box.style.cssText='font-style:italic;font-family:inherit';\n      box.textContent='\\u23f3 Server processing previous frame\\u2026';\n      pendingFrame=false;setTimeout(sendFrame,CONFIG.FRAME_BUSY_RETRY);return;\n    }\n\n    const score=data.score||0;\n    const low=data.low_conf||false;\n    updateConfMeter(score);\n\n    // \u2500\u2500 Current frame output \u2500\u2500\n    const currentText=data.current_text||'';\n    const textChanged=currentText&&currentText!==lastCamText&&!currentText.includes('[No text');\n    const isHighConf=score>=CONFIG.HIGH_CONF_SCORE;\n\n    if(textChanged){\n      if(isHighConf)highConfFrames++;else highConfFrames=0;\n      sameTextCount=0;lastCamText=currentText;currentScore=score;\n      lockedOnText=false;\n      document.getElementById('lock-badge').style.display='none';\n      document.getElementById('stab-fill').style.width='0%';\n      box.className='output-box fade-in';\n      box.style.color=low?'var(--warn)':'var(--accent2)';\n      box.style.fontStyle='normal';\n      box.style.fontFamily=\"'Courier New',monospace\";\n      box.innerHTML=buildDiffHTML(lastCamText||'',currentText);\n      renderInsight('cam-insight',currentText);\n    }else{\n      sameTextCount++;\n      const stab=Math.min(100,(sameTextCount/CONFIG.STABLE_LOCK_THRESHOLD)*100);\n      document.getElementById('stab-fill').style.width=stab+'%';\n      if(sameTextCount>=CONFIG.STABLE_LOCK_THRESHOLD&&highConfFrames>=CONFIG.HIGH_CONF_FRAMES_REQ&&!lockedOnText){\n        lockedOnText=true;\n        document.getElementById('lock-badge').style.display='';\n      }\n    }\n    if(!currentText||currentText.startsWith('[')){\n      box.className='output-box empty';\n      box.style.fontStyle='italic';\n      box.style.fontFamily='inherit';\n      box.textContent=currentText||'[No text found \u2014 try better lighting or hold device steady]';\n    }\n\n    // \u2500\u2500 Accumulated \u2014 straight from server (only render if not editing) \u2500\u2500\n    if(accumulateEnabled){\n      const prevAcc=serverAccumulated;\n      const incoming=data.accumulated_text||'';\n      // Always update serverAccumulated (even if not rendering, so Edit has fresh data)\n      serverAccumulated=incoming;\n      if(incoming!==prevAcc){\n        renderAccumulated(incoming,prevAcc);\n      }\n    }\n\n    document.getElementById('camera-meta').innerHTML=\n      `<div class=\"meta-chip\">&#9201; <span>${data.elapsed}s</span></div>`+\n      `<div class=\"meta-chip\">Frame <span>#${data.frame_id}</span></div>`+\n      `<div class=\"meta-chip\">Score <span>${score}</span></div>`+\n      `<div class=\"meta-chip\">${!low&&currentText&&!currentText.startsWith('[')&&textChanged?'&#128994; New':'&#128309; Same'}</div>`+\n      `<div class=\"meta-chip\">&#128421; <span>${data.device||'?'}</span></div>`;\n    document.getElementById('frame-info').textContent=`Frames: ${frameCount} | ${data.elapsed}s`;\n\n    pendingFrame=false;\n    setStatus('cam-status',false,'&#9989; Connected \\u2014 live OCR active');\n\n    const delay=lockedOnText?CONFIG.ADAPTIVE_SLOW_DELAY:\n                sameTextCount>=CONFIG.ADAPTIVE_SAME_LIMIT?CONFIG.ADAPTIVE_SLOW_DELAY:\n                CONFIG.FRAME_SEND_DELAY;\n    setTimeout(sendFrame,delay);\n  };\n\n  cameraWS.onerror=()=>{setStatus('cam-status',false,'&#10060; WebSocket error');pendingFrame=false;};\n  cameraWS.onclose=()=>{\n    pendingFrame=false;\n    if(camState===CAM_STATE.STOPPED)return;\n    const delays=CONFIG.WS_RECONNECT_DELAYS;\n    if(wsReconnectAttempt>=delays.length){setCamState(CAM_STATE.RECONNECTING);setStatus('cam-status',false,'&#10060; Connection failed. Stop and restart.');return;}\n    const delay=delays[wsReconnectAttempt++];\n    setCamState(CAM_STATE.RECONNECTING);\n    setStatus('cam-status',true,`Reconnecting in ${delay/1000}s (attempt ${wsReconnectAttempt}/${delays.length})\\u2026`);\n    wsReconnectTimer=setTimeout(connectCameraWS,delay);\n  };\n}\n\nfunction sendFrame(){\n  if(!cameraWS||cameraWS.readyState!==WebSocket.OPEN)return;\n  if(pendingFrame)return;\n  if(!navigator.onLine)return;\n  const video=document.getElementById('camera-video');\n  const canvas=document.getElementById('camera-canvas');\n  if(!video||video.readyState<2||video.videoWidth===0){setTimeout(sendFrame,300);return;}\n  canvas.width=video.videoWidth;canvas.height=video.videoHeight;\n  canvas.getContext('2d').drawImage(video,0,0);\n  const hasMotion=checkMotion(canvas);\n  if(lockedOnText&&!hasMotion){setTimeout(sendFrame,500);return;}\n  if(hasMotion&&lockedOnText){\n    lockedOnText=false;sameTextCount=0;highConfFrames=0;currentScore=0;\n    document.getElementById('lock-badge').style.display='none';\n    document.getElementById('stab-fill').style.width='0%';\n  }\n  const b64=canvas.toDataURL('image/jpeg',CONFIG.CAM_JPEG_QUALITY);\n  const hash=frameHash(b64);\n  if(hash===lastFrameHash){setTimeout(sendFrame,CONFIG.FRAME_SEND_DELAY*4);return;}\n  lastFrameHash=hash;\n  frameCount++;pendingFrame=true;\n  setStatus('cam-status',true,'Processing frame\\u2026');\n  // v13: include accumulate flag\n  cameraWS.send(JSON.stringify({frame:b64,accumulate:accumulateEnabled}));\n  setTimeout(()=>{if(pendingFrame){pendingFrame=false;sendFrame();}},CONFIG.FRAME_SAFETY_TIMEOUT);\n}\n\nfunction stopCamera(){\n  setCamState(CAM_STATE.STOPPED);\n  if(wsReconnectTimer){clearTimeout(wsReconnectTimer);wsReconnectTimer=null;}\n  if(cameraWS){cameraWS.close();cameraWS=null;}\n  if(cameraStream){cameraStream.getTracks().forEach(t=>t.stop());cameraStream=null;}\n  document.getElementById('camera-video').srcObject=null;\n  document.getElementById('cam-start-btn').style.display='';\n  document.getElementById('cam-stop-btn').style.display='none';\n  document.getElementById('live-badge').style.display='none';\n  document.getElementById('lock-badge').style.display='none';\n  document.getElementById('conf-bar-wrap').style.display='none';\n  frameCount=0;pendingFrame=false;lastCamText='';lastFrameHash='';\n  sameTextCount=0;lockedOnText=false;lastFramePixels=null;\n  currentScore=0;highConfFrames=0;\n  document.getElementById('frame-info').textContent='';\n  document.getElementById('cam-insight').classList.remove('show');\n  document.getElementById('stab-fill').style.width='0%';\n  document.getElementById('motion-dot').classList.remove('active');\n  setStatus('cam-status',false,'&#128247; Camera stopped');\n  setCamState(CAM_STATE.IDLE);\n}\n\n// \u2500\u2500\u2500 HISTORY \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction saveToHistory(filename,text,elapsed,thumbUrl){\n  const history=JSON.parse(sessionStorage.getItem('ocrHistory')||'[]');\n  history.unshift({filename,text,elapsed,time:Date.now(),thumbUrl});\n  if(history.length>CONFIG.HISTORY_MAX)history.pop();\n  sessionStorage.setItem('ocrHistory',JSON.stringify(history));\n}\nfunction loadHistory(){return JSON.parse(sessionStorage.getItem('ocrHistory')||'[]');}\nfunction clearHistory(){sessionStorage.removeItem('ocrHistory');renderHistory();}\nfunction renderHistory(){\n  const list=document.getElementById('history-list');\n  const history=loadHistory();\n  if(!history.length){list.innerHTML='<div style=\"color:var(--muted);font-size:.86rem;padding:12px 0\">No history yet.</div>';return;}\n  list.innerHTML=history.map((h,i)=>{\n    const ago=formatAgo(h.time),preview=(h.text||'').replace(/\\n/g,' ').slice(0,120);\n    return `<div class=\"history-item\" onclick=\"restoreHistory(${i})\">\n      <div class=\"history-thumb\" style=\"${h.thumbUrl?`background:url(${h.thumbUrl}) center/cover`:''}\">\n        ${h.thumbUrl?'':'&#128444;&#65039;'}\n      </div>\n      <div class=\"history-meta\">\n        <div class=\"history-name\">${esc(h.filename)}</div>\n        <div class=\"history-preview\">${esc(preview)}</div>\n        <div class=\"history-time\">${ago} &middot; ${h.elapsed}s</div>\n      </div>\n    </div>`;\n  }).join('');\n}\nfunction restoreHistory(i){\n  const h=loadHistory()[i];if(!h)return;\n  switchTab('image');\n  const box=document.getElementById('output-text');\n  box.className='output-box';box.style.color='var(--accent2)';box.textContent=h.text;\n  document.getElementById('img-export-bar').style.display='';\n  lastSingleResult={text:h.text,elapsed:h.elapsed,score:'(history)',device:''};\n  renderInsight('img-insight',h.text);\n  showToast('Restored from history');\n}\nfunction formatAgo(ts){\n  const s=Math.floor((Date.now()-ts)/1000);\n  if(s<60)return s+'s ago';if(s<3600)return Math.floor(s/60)+'m ago';return Math.floor(s/3600)+'h ago';\n}\n\n// \u2500\u2500\u2500 INSIGHT \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction renderInsight(panelId,text){\n  const panel=document.getElementById(panelId);\n  if(!text||text.includes('[No text')){panel.classList.remove('show');return;}\n  const lines=text.split('\\n').map(l=>l.trim()).filter(Boolean);\n  const words=text.split(/\\s+/).filter(Boolean).length;\n  const chars=text.replace(/\\s/g,'').length;\n  const tags=[];\n  const hasCurr=/[\\u20ac$\\u00a3\\u00a5\\u20b9]|\\b(total|subtotal|tax|vat|invoice|receipt|amount|due|paid|price|rs\\.?|inr)\\b/i.test(text);\n  const hasAmts=/\\d+[.,]\\d{2}/.test(text);\n  if(hasCurr&&hasAmts)tags.push({cls:'itag-receipt',label:'&#129534; Receipt / Invoice'});\n  const phones=(text.match(/[+]?[\\d\\s\\-(). ]{10,}/g)||[]).filter(p=>p.replace(/\\D/g,'').length>=9);\n  if(phones.length)tags.push({cls:'itag-contact',label:'&#128222; Phone Number'});\n  if(/[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-z]{2,}/.test(text))tags.push({cls:'itag-contact',label:'&#9993;&#65039; Email'});\n  if(/https?:\\/\\/|www\\./i.test(text))tags.push({cls:'itag-url',label:'&#128279; URL'});\n  if(/\\b(\\d{1,2}[\\/\\-]\\d{1,2}[\\/\\-]\\d{2,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\\b/i.test(text))tags.push({cls:'itag-date',label:'&#128197; Date'});\n  if(/\\b([A-Z]{1,3}[-\\s]?\\d{4,}|\\d{4}[\\s-]\\d{4}[\\s-]\\d{4}|passport|aadhar|pan\\s|license|id\\s*no|roll\\s*no)\\b/i.test(text))tags.push({cls:'itag-id',label:'&#129706; ID / Card'});\n  if(!tags.length&&/^[\\d\\s.,+\\-]+$/.test(text.trim()))tags.push({cls:'itag-number',label:'&#128290; Numeric'});\n  if(!tags.length&&words>10)tags.push({cls:'itag-doc',label:'&#128196; Document'});\n  const amounts=[...text.matchAll(/[\\u20ac$\\u00a3\\u20b9\\u00a5]\\s*[\\d,.]+|[\\d,.]+\\s*[\\u20ac$\\u00a3\\u20b9\\u00a5]/g)].map(m=>m[0]).slice(0,4);\n  const emails=[...text.matchAll(/[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-z]{2,}/g)].map(m=>m[0]).slice(0,2);\n  const urls=[...text.matchAll(/https?:\\/\\/[^\\s]+/g)].map(m=>m[0]).slice(0,1);\n  function chip(label,val,copyVal){\n    const cv=JSON.stringify(copyVal||val);\n    return `<div class=\"insight-chip\"><div class=\"ic-label\">${label}</div><div class=\"ic-val\">${esc(val)}</div>${copyVal!==undefined?`<button class=\"ic-copy\" onclick=\"quickCopy(${cv})\" title=\"Copy\">&#128203;</button>`:''}</div>`;\n  }\n  let chips=chip('Words',words.toLocaleString())+chip('Characters',chars.toLocaleString())+chip('Lines',lines.length);\n  if(amounts.length)chips+=chip('Amounts',amounts.join(', '),amounts.join(', '));\n  if(emails.length)chips+=chip('Email',emails[0],emails[0]);\n  if(urls.length)chips+=chip('URL',urls[0].slice(0,36)+(urls[0].length>36?'\\u2026':''),urls[0]);\n  const tagsHTML=tags.map(t=>`<span class=\"itag ${t.cls}\">${t.label}</span>`).join('');\n  panel.innerHTML=`<div class=\"insight-header\">&#128161; Instant Insight</div><div class=\"insight-grid\">${chips}</div>${tagsHTML?`<div class=\"insight-tags\">${tagsHTML}</div>`:''}`;\n  panel.classList.add('show');\n}\n\n// \u2500\u2500\u2500 UTILITIES \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nfunction setStatus(id,loading,msg){document.getElementById(id).innerHTML=msg?`${loading?'<div class=\"spinner\"></div>':''}<span>${msg}</span>`:'';}\nfunction esc(s){const d=document.createElement('div');d.textContent=s||'';return d.innerHTML;}\nfunction copyOutput(){copyText(document.getElementById('output-text').textContent);}\nfunction copyCameraOutput(){copyText(document.getElementById('camera-output').textContent);}\nfunction quickCopy(val){copyText(val);}\nfunction copyText(text,btn){\n  navigator.clipboard.writeText(text).then(()=>{\n    showToast('Copied!');\n    if(btn){btn.classList.add('copied');btn.textContent='Copied!';setTimeout(()=>{btn.classList.remove('copied');btn.textContent='Copy';},2000);}\n  });\n}\nfunction showToast(msg){const t=document.getElementById('toast');t.textContent=msg;t.classList.add('show');setTimeout(()=>t.classList.remove('show'),2000);}\n\n// \u2500\u2500\u2500 HEALTH CHECK \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n(async function checkHealth(){\n  try{\n    const r=await fetch(`${API}/health`);const d=await r.json();\n    const b=document.getElementById('gpu-badge');\n    const isCPU=d.gpu.includes('CPU');\n    b.textContent=(isCPU?'\\u26a0\\ufe0f ':'\\u2705 ')+d.gpu;\n    b.className='badge'+(isCPU?' warn':'');\n    if(isCPU)document.getElementById('cpu-warn').classList.add('show');\n    const rlb=document.getElementById('rl-badge');\n    if(d.rate_limits){rlb.textContent=`&#128444;&#65039; ${d.rate_limits.image_remaining} req left`;rlb.style.display='';}\n  }catch{\n    const b=document.getElementById('gpu-badge');\n    b.textContent='\\u26a0\\ufe0f Server offline';b.className='badge err';\n  }\n})();\n</script>\n</body>\n</html>"

pathlib.Path("static").mkdir(exist_ok=True)
pathlib.Path("static/index.html").write_text(html_code)
print("✅ static/index.html written (v13.1)")
print("✅ All files ready — run Cell 5")


In [ ]:
#@title 4️⃣b  🔧 Patch index.html — Fix Edit Bug + Add HTTP Camera Mode
# Fix 1: Edit textarea was invisible (CSS display:none not overridden)
# Fix 2: Add HTTP REST camera mode as fallback when WebSocket fails
import pathlib

html_path = pathlib.Path("static/index.html")
html = html_path.read_text()

# ─── FIX 1: Edit button textarea visibility bug ───────────────────────────────
# startEditAcc() called editArea.style.display='' which just removes the inline
# style, leaving the CSS class .acc-edit-area { display:none } in effect.
# Result: textarea stays hidden, all accumulated text appears to vanish.
# Fix: explicitly set display='block'.
assert "editArea.style.display='';" in html, "Fix1: target not found"
html = html.replace("editArea.style.display='';", "editArea.style.display='block';", 1)
print("\u2705 Fix 1: Edit textarea now uses display='block' — textarea is visible on edit click")

# ─── FIX 2: HTTP Mode button + badge in camera controls ──────────────────────
old_fi = '<span class="frame-info" id="frame-info"></span>'
new_fi = (
    old_fi
    + '\n          <button class="btn btn-ghost btn-sm" id="cam-http-btn"'
    + ' onclick="switchToHttpMode()"'
    + ' title="WebSocket unavailable? Click to use HTTP REST endpoint instead"'
    + ' style="display:none">&#9889; Use HTTP Mode</button>'
    + '\n          <span class="cam-state-badge" id="cam-api-badge"'
    + ' style="display:none;border-color:var(--cyan);color:var(--cyan)">&#9889; HTTP Mode</span>'
)
assert old_fi in html, "Fix2: frame-info target not found"
html = html.replace(old_fi, new_fi, 1)
print("\u2705 Fix 2: HTTP mode button added to camera controls bar")

# ─── FIX 3: WS onclose — expose HTTP fallback button after all retries ────────
old3 = (
    "setCamState(CAM_STATE.RECONNECTING);"
    "setStatus('cam-status',false,'&#10060; Connection failed. Stop and restart.');"
    "return;}"
)
new3 = (
    "setCamState(CAM_STATE.RECONNECTING);"
    "setStatus('cam-status',false,'&#10060; WS failed \u2014 click &#9889; Use HTTP Mode for REST fallback');"
    "document.getElementById('cam-http-btn').style.display='';"
    "return;}"
)
assert old3 in html, "Fix3: WS onclose target not found"
html = html.replace(old3, new3, 1)
print("\u2705 Fix 3: After all WS retries fail, HTTP Mode button is revealed automatically")

# ─── FIX 4: stopCamera — clear HTTP timer and reset mode badge ───────────────
old4 = "function stopCamera(){"
new4 = (
    "function stopCamera(){"
    "if(httpCameraTimer){clearTimeout(httpCameraTimer);httpCameraTimer=null;}"
    "cameraApiMode='ws';"
    "document.getElementById('cam-api-badge').style.display='none';"
    "document.getElementById('cam-http-btn').style.display='none';"
)
assert old4 in html, "Fix4: stopCamera target not found"
html = html.replace(old4, new4, 1)
print("\u2705 Fix 4: stopCamera cleans up HTTP timer and resets mode badge")

# ─── FIX 5: HTTP camera JS (variables + switchToHttpMode + sendFrameHTTP) ────
HTTP_JS = """
// \u2500\u2500\u2500 HTTP CAMERA MODE (REST fallback when WebSocket is unavailable) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
// Instead of a persistent WebSocket session, the client POST-s each frame to
// /ocr/camera/frame and receives (current_text, accumulated_text) back.
// Accumulated text is managed client-side: sent with every request so the
// server can merge and return the updated version (stateless HTTP pattern).
let cameraApiMode  = 'ws';   // 'ws' | 'http'
let httpCameraTimer = null;

function switchToHttpMode() {
  if (cameraApiMode === 'http') return;
  cameraApiMode = 'http';
  if (wsReconnectTimer) { clearTimeout(wsReconnectTimer); wsReconnectTimer = null; }
  if (cameraWS) { try { cameraWS.close(); } catch(e){} cameraWS = null; }
  document.getElementById('cam-http-btn').style.display  = 'none';
  document.getElementById('cam-api-badge').style.display = '';
  setCamState(CAM_STATE.ACTIVE);
  setStatus('cam-status', false, '\\u26a1 HTTP mode \\u2014 posting frames to /ocr/camera/frame');
  pendingFrame = false;
  sendFrameHTTP();
}

async function sendFrameHTTP() {
  if (cameraApiMode !== 'http') return;
  if (!cameraStream) return;
  const video  = document.getElementById('camera-video');
  const canvas = document.getElementById('camera-canvas');
  if (!video || video.readyState < 2 || video.videoWidth === 0) {
    httpCameraTimer = setTimeout(sendFrameHTTP, 500); return;
  }
  canvas.width  = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);
  const hasMotion = checkMotion(canvas);
  if (lockedOnText && !hasMotion) { httpCameraTimer = setTimeout(sendFrameHTTP, 1200); return; }
  if (hasMotion && lockedOnText) {
    lockedOnText = false; sameTextCount = 0; highConfFrames = 0;
    document.getElementById('lock-badge').style.display  = 'none';
    document.getElementById('stab-fill').style.width     = '0%';
  }
  canvas.toBlob(async (blob) => {
    if (pendingFrame) { httpCameraTimer = setTimeout(sendFrameHTTP, 600); return; }
    pendingFrame = true;
    setStatus('cam-status', true, '\\u23f3 Processing frame\\u2026');
    frameCount++;
    const fd = new FormData();
    fd.append('file',        blob,   'frame.jpg');
    fd.append('accumulated', accumulateEnabled ? (serverAccumulated || '') : '');
    try {
      const res = await fetch(`${API}/ocr/camera/frame`, { method: 'POST', body: fd });
      if (res.status === 429) {
        const d = await res.json();
        document.getElementById('cam-rl-banner').textContent = '\\u23f1 ' + d.error;
        document.getElementById('cam-rl-banner').classList.add('show');
        pendingFrame = false; httpCameraTimer = setTimeout(sendFrameHTTP, 3500); return;
      }
      document.getElementById('cam-rl-banner').classList.remove('show');
      const data = await res.json();
      _handleCamResult(data);
    } catch (err) {
      setStatus('cam-status', false, '\\u274c HTTP error: ' + err.message);
    }
    pendingFrame = false;
    const delay = lockedOnText ? 3000
                : sameTextCount >= CONFIG.ADAPTIVE_SAME_LIMIT ? CONFIG.ADAPTIVE_SLOW_DELAY : 2000;
    httpCameraTimer = setTimeout(sendFrameHTTP, delay);
  }, 'image/jpeg', CONFIG.CAM_JPEG_QUALITY);
}

// Shared result-handler used by BOTH WebSocket and HTTP camera modes.
// Extracted from the WS onmessage handler so there is one code path.
function _handleCamResult(data) {
  if (data.error) { setStatus('cam-status', false, '\\u26a0\\ufe0f ' + data.error); return; }
  const box   = document.getElementById('camera-output');
  const score = data.score    || 0;
  const low   = data.low_conf || false;
  updateConfMeter(score);
  const currentText = data.current_text || '';
  const textChanged = currentText && currentText !== lastCamText && !currentText.includes('[No text');
  const isHighConf  = score >= CONFIG.HIGH_CONF_SCORE;
  if (textChanged) {
    if (isHighConf) highConfFrames++; else highConfFrames = 0;
    sameTextCount = 0; lastCamText = currentText; currentScore = score; lockedOnText = false;
    document.getElementById('lock-badge').style.display  = 'none';
    document.getElementById('stab-fill').style.width     = '0%';
    box.className      = 'output-box fade-in';
    box.style.color    = low ? 'var(--warn)' : 'var(--accent2)';
    box.style.fontStyle  = 'normal';
    box.style.fontFamily = "\\'Courier New\\',monospace";
    box.innerHTML = buildDiffHTML(lastCamText || '', currentText);
    renderInsight('cam-insight', currentText);
  } else {
    sameTextCount++;
    const stab = Math.min(100, (sameTextCount / CONFIG.STABLE_LOCK_THRESHOLD) * 100);
    document.getElementById('stab-fill').style.width = stab + '%';
    if (sameTextCount >= CONFIG.STABLE_LOCK_THRESHOLD
        && highConfFrames >= CONFIG.HIGH_CONF_FRAMES_REQ && !lockedOnText) {
      lockedOnText = true; document.getElementById('lock-badge').style.display = '';
    }
  }
  if (!currentText || currentText.startsWith('[')) {
    box.className       = 'output-box empty';
    box.style.fontStyle  = 'italic';
    box.style.fontFamily = 'inherit';
    box.textContent     = currentText || '[No text found \\u2014 try better lighting or hold steady]';
  }
  if (accumulateEnabled) {
    const prevAcc  = serverAccumulated;
    const incoming = data.accumulated_text || '';
    serverAccumulated = incoming;
    if (incoming !== prevAcc) renderAccumulated(incoming, prevAcc);
  }
  document.getElementById('camera-meta').innerHTML =
    `<div class="meta-chip">&#9201; <span>${data.elapsed}s</span></div>` +
    `<div class="meta-chip">Frame <span>#${frameCount}</span></div>` +
    `<div class="meta-chip">Score <span>${score}</span></div>` +
    `<div class="meta-chip">${!low && currentText && !currentText.startsWith('[') && textChanged
        ? '&#128994; New' : '&#128309; Same'}</div>` +
    `<div class="meta-chip">&#128421; <span>${data.device || '?'}</span></div>`;
  document.getElementById('frame-info').textContent = `Frames: ${frameCount} | ${data.elapsed}s`;
  setStatus('cam-status', false,
    cameraApiMode === 'http' ? '\\u26a1 HTTP mode active' : '\\u2705 Connected \\u2014 live OCR active');
}
"""

cam_marker = "// \u2500\u2500\u2500 CAMERA \u2500"
assert cam_marker in html, f"Fix5: camera marker not found ({repr(cam_marker)})"
html = html.replace(cam_marker, HTTP_JS + "\n" + cam_marker, 1)
print("\u2705 Fix 5: HTTP camera JS added (switchToHttpMode / sendFrameHTTP / _handleCamResult)")

html_path.write_text(html)
print("\n\u2705 static/index.html fully patched!")
print("  \u2022 Edit button: textarea is now visible (display:block)")
print("  \u2022 Camera: &#9889; Use HTTP Mode button appears when WS fails all retries")
print("  \u2022 Camera: HTTP mode posts frames to POST /ocr/camera/frame")
print("  \u2022 Camera: accumulated text managed client-side (stateless HTTP pattern)")


In [ ]:
#@title 5️⃣ Start Server (Cloudflare Tunnel + uvicorn) — v13
import subprocess, threading, time, os, urllib.request, stat, re
import nest_asyncio, asyncio, uvicorn
nest_asyncio.apply()

PORT = 8000

CF_BIN = '/tmp/cloudflared'
if not os.path.exists(CF_BIN):
    print('⬇️  Downloading cloudflared...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CF_BIN)
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)
    print('✅ cloudflared downloaded')
else:
    print('✅ cloudflared already present')

cf_log = '/tmp/cf_tunnel.log'
open(cf_log, 'w').close()
subprocess.Popen([CF_BIN, 'tunnel', '--url', f'http://localhost:{PORT}'],
                 stdout=open(cf_log,'w'), stderr=subprocess.STDOUT)

PUBLIC_URL = None

def watch_log():
    global PUBLIC_URL
    printed = False
    while True:
        try:
            urls = re.findall(r'https://[a-z0-9\-]+\.trycloudflare\.com', open(cf_log).read())
            if urls and not printed:
                PUBLIC_URL = urls[0]
                printed = True
                print('\n' + '═'*62)
                print(f'  🔗 PUBLIC URL : {PUBLIC_URL}')
                print(f'  📄 Swagger UI : {PUBLIC_URL}/docs')
                print(f'  📄 ReDoc      : {PUBLIC_URL}/redoc')
                print('  Open the PUBLIC URL in your browser!')
                print('═'*62 + '\n')
        except: pass
        time.sleep(1)

threading.Thread(target=watch_log, daemon=True).start()
print(f'🌐 Cloudflare Tunnel starting... waiting for URL (10–20s)\n')
time.sleep(6)  # wait a bit longer so PUBLIC_URL is captured before server starts

# ── Key fix: set ROOT_PATH so FastAPI /docs works behind Cloudflare ──────────
# FastAPI uses root_path to build the OpenAPI spec URL inside Swagger UI.
# Without it, the browser requests /openapi.json using the wrong base URL.
if PUBLIC_URL:
    os.environ['ROOT_PATH'] = PUBLIC_URL
    print(f'✅ ROOT_PATH set to: {PUBLIC_URL}')
else:
    os.environ['ROOT_PATH'] = ''
    print('⚠️  ROOT_PATH not set (tunnel URL not captured yet) — /docs may not load correctly')

print('🚀 Starting OCR API server (v13)...')
config = uvicorn.Config('main:app', host='0.0.0.0', port=PORT, log_level='warning')
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())
